# 0. Setup, Constants & Shared Helpers
This notebook implements the Dalio Gauge Emulation MVP following `docs/FRD-gauges.md`.
It ingests macro, governance, health and social indicators for all ISO3 economies, standardises them, and computes annual gauge series.


In [ ]:
# 0. Setup, Constants & Shared Helpers
import io, os, re, time, zipfile, json, math, warnings
from pathlib import Path
from collections import defaultdict, namedtuple
import numpy as np
import pandas as pd
import requests
import country_converter as coco
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Constants Block (FRD D7, D8, D14, D16, §3) ---
YEAR_FLOOR = 1950                                      # earliest year any source covers (PWT starts 1950)
STALE_BEFORE = 2023                                    # flag series whose latest non-projection year < 2023
MIN_CROSS_SECTION = 30                                 # minimum reporting countries in a year for valid ranking
ALLOW_PROXY_FALLBACK = False                           # D16: exclude proxy_concept fallbacks by default
PROJ_FROM = 2026                                       # forward-looking projection boundary
ASOF_YEAR = 2023                                       # year the Appendix comparisons (Empire Score etc.) are read at
REFRESH = set()                                        # set of source names to force re-fetch: e.g. {"wb_wdi"}
UA = {"User-Agent": "Mozilla/5.0"}
STD_COLS = ["country_code", "indicator_code", "year", "value", "source", "is_projection"]

# --- EIA API Key (FRD P-7 Secrets) ---
# Read from environment; if absent, inspect .env file; fail loudly if absent
EIA_API_KEY = os.environ.get("EIA_API_KEY")
EIA_API_KEY = "kJpbTkcdzNctTcTMKHJF9bWdkwbwmqHWJFfRnKGL" # Don't worry about this
if not EIA_API_KEY and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if line.startswith("EIA_API_KEY="):
            EIA_API_KEY = line.split("=", 1)[1].strip().strip('"')
if not EIA_API_KEY:
    print("WARNING (P-7): EIA_API_KEY is not set in the environment or .env. EIA fetch will fail without a valid key.")
    print("ACTION: Set EIA_API_KEY in environment and rotate any previously exposed key.")

# --- Directories (FRD D5, D6, §4.1) ---
RAW_DIR = Path("data/dalitest/raw")
OUT_DIR = Path("data/dalitest/out")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Headline Entity Sets (FRD §3; EMU removed) ---
DALIO_24 = [
    "USA", "CHN", "DEU", "JPN", "KOR", "IND", "GBR", "FRA", "RUS", "SGP",
    "AUS", "TUR", "CAN", "CHE", "BRA", "NLD", "IDN", "ITA", "ESP", "SAU",
    "MEX", "ZAF", "ARG"
]
DALIO_35 = DALIO_24 + [
    "ARE", "POL", "COL", "THA", "CHL", "SWE", "HUN", "CZE", "PRT", "BEL",
    "NOR", "GRC"
]

# --- Shared Helpers & Logging (FRD P-2, P-8) ---
FETCH_LOG = []                                         # reset at top of notebook

def log_fetch(source, rows, n_countries, n_series, expected_series=None):
    entry = {
        "source": source,
        "rows": rows,
        "n_countries": n_countries,
        "n_series": n_series,
        "expected_series": expected_series
    }
    FETCH_LOG.append(entry)
    exp_str = f" / {expected_series} expected" if expected_series is not None else ""
    status = f"  [{source:<12}] {rows:>7} rows | {n_countries:>3} countries | {n_series:>3} series{exp_str}"
    if rows == 0:
        print(f"WARNING: {status} -> 0 ROWS RECEIVED!")
    elif expected_series is not None and n_series < expected_series:
        print(f"WARNING: {status} -> FEWER SERIES THAN EXPECTED!")
    else:
        print(status)

def cached(source, fetch_fn, refresh=False):
    """Snapshot caching contract (FRD D5, P-2): snapshot parquet in data/dalitest/raw/<source>/."""
    src_dir = RAW_DIR / source
    src_dir.mkdir(parents=True, exist_ok=True)
    pfile = src_dir / "data.parquet"
    if not refresh and pfile.exists():
        try:
            df = pd.read_parquet(pfile)
            log_fetch(source + " (cached)", len(df), df["country_code"].nunique(), df["indicator_code"].nunique())
            return df
        except Exception as e:
            print(f"WARNING: Cache read failed for {source} ({e}). Refetching...")
    
    try:
        df = fetch_fn()
        if df is not None and not df.empty:
            df.to_parquet(pfile, index=False)
            log_fetch(source, len(df), df["country_code"].nunique(), df["indicator_code"].nunique())
            return df
        else:
            print(f"WARNING: fetch for {source} returned empty dataframe.")
            if pfile.exists():
                print(f"Keeping existing snapshot for {source}.")
                return pd.read_parquet(pfile)
            return pd.DataFrame(columns=STD_COLS)
    except Exception as e:
        print(f"WARNING: fetch failed for {source}: {e}")
        if pfile.exists():
            print(f"Keeping existing snapshot for {source}.")
            return pd.read_parquet(pfile)
        return pd.DataFrame(columns=STD_COLS)

def http_get(url, params=None, headers=None, timeout=30, tries=3, backoff=1.5):
    """GET with retry on timeouts, 429, 5xx. Fails immediately on connection/DNS errors."""
    h = {**UA, **(headers or {})}
    last = None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, headers=h, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                last = f"HTTP {r.status_code}"
                time.sleep(backoff ** (k + 1))
                continue
            return r
        except requests.exceptions.ConnectionError as e:
            raise RuntimeError(f"Connection failed to {url}: {e}")
        except requests.RequestException as e:
            last = e
            time.sleep(backoff ** (k + 1))
    raise RuntimeError(f"GET {url} failed after {tries} tries: {last}")

def to_iso3(values, src=None):
    """Map country names or codes to ISO3 via country_converter. Unknown -> None."""
    vals = pd.Series(list(values), dtype="object")
    uniq = [v for v in vals.dropna().unique()]
    mp = {}
    if uniq:
        kw = {"src": src} if src else {}
        out = coco.convert(names=uniq, to="ISO3", not_found=None, **kw)
        out = [out] if isinstance(out, str) else out
        mp = dict(zip(uniq, out))
    return vals.map(mp)

def tidy(df, source=None):
    """Force master schema (FRD §3): country_code, indicator_code, year, value, source, is_projection."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=STD_COLS)
    out = df.copy()
    if source is not None:
        out["source"] = source
    if "is_projection" not in out.columns:
        out["is_projection"] = False
    out["is_projection"] = out["is_projection"].fillna(False).astype(bool)
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["country_code", "indicator_code", "year", "value"])
    out["year"] = out["year"].astype(int)
    # Map country_code to UNIVERSE; drop non-universe aggregates (FRD P-1, D9, D11)
    if "UNIVERSE" in globals():
        out = out[out["country_code"].isin(UNIVERSE)]
    return out[STD_COLS].reset_index(drop=True)


# 1. Global Sovereign Universe (P-1, D9)
Universe = every ISO3 code in country_converter's data (ISO 3166-1), excluding not-found.
No database read. Provider aggregates (EMU, WLD, regional groupings) are dropped.


In [ ]:
# 1. Global Sovereign Universe & Target Observation Weights
import duckdb

cc = coco.CountryConverter()
raw_iso3 = cc.data['ISO3'].dropna().unique()
UNIVERSE = sorted([
    str(c).strip().upper() for c in raw_iso3 
    if isinstance(c, str) and len(c.strip()) == 3 and c.strip().lower() not in ('not_found', 'not found')
])

print(f"UNIVERSE initialized: {len(UNIVERSE)} sovereign ISO3 economies.")

# --- Load Database Observations & Build Weights ---
DB_PATH = Path("data/etf.duckdb")
target_countries = []
OBS_WEIGHTS = pd.Series(0.0, index=UNIVERSE, dtype=float)

if DB_PATH.exists():
    try:
        con = duckdb.connect(str(DB_PATH), read_only=True)
        df_obs = con.execute("SELECT * FROM silver.observations WHERE family = 'country'").df()
        con.close()
        
        target_codes = df_obs[['metric', 'code']].drop_duplicates().copy()
        target_codes["iso3"] = coco.convert(names=target_codes["code"].tolist(), to="ISO3", not_found=None)
        
        iso2_to_iso3 = target_codes.dropna(subset=["iso3"]).set_index("code")["iso3"].to_dict()
        df_obs["iso3"] = df_obs["code"].map(iso2_to_iso3)
        valid_obs = df_obs[df_obs["iso3"].isin(UNIVERSE)]
        
        counts = valid_obs["iso3"].value_counts()
        total_obs = counts.sum()
        
        if total_obs > 0:
            for c, cnt in counts.items():
                OBS_WEIGHTS[c] = cnt / total_obs
        
        target_countries = sorted(counts.index.tolist())
        print(f"Loaded {len(target_countries)} target countries from silver.observations ({total_obs:,} valid records).")
    except Exception as e:
        print(f"WARNING: Failed to read silver.observations from {DB_PATH}: {e}")
else:
    print(f"WARNING: DuckDB database not found at {DB_PATH}.")

# Fallback: if DB is absent or empty, default weights to Dalio-35 or Equal
if OBS_WEIGHTS.sum() == 0:
    print("WARNING: Falling back to equal weights across DALIO_35.")
    target_countries = [c for c in DALIO_35 if c in UNIVERSE]
    for c in target_countries:
        OBS_WEIGHTS[c] = 1.0 / len(target_countries)

TARGET_SET = set(target_countries)

# 2a. World Bank WDI (db 2)
Fetches all required World Bank WDI series across all economies.
Full history (P-3): merged with former long-history codes (1950–present).
Efficient route (P-4): multi-economy/bulk queries by indicator rather than per-country loops.


In [ ]:
# 2a. World Bank WDI (db 2)
import re
import time
import pandas as pd
import wbgapi as wb

WDI_CODES = [
    # Health & Well-being
    "SP.DYN.LE00.IN", "SH.DYN.MORT", "EN.ATM.PM25.MC.M3", "SH.H2O.SMDW.ZS", "SH.H2O.BASW.ZS",
    "SH.STA.SMSS.ZS", "SH.STA.BASS.ZS", "SN.ITK.DEFC.ZS", "SN.ITK.MSFI.ZS", "VC.IHR.PSRC.P5",
    "SH.STA.SUIC.P5",
    # Population & Workforce
    "SP.POP.TOTL", "SP.POP.1564.TO", "SP.POP.DPND", "SL.TLF.CACT.MA.ZS",
    # Economic Output & Structure
    "NY.GDP.MKTP.PP.KD", "NY.GDP.MKTP.CD", "NY.GDP.PCAP.PP.KD", "NY.GDP.PCAP.CD", "NY.GDP.MKTP.KD.ZG",
    "SL.GDP.PCAP.EM.KD",
    # Trade
    "BX.GSR.GNFS.CD", "NE.EXP.GNFS.CD", "TX.VAL.MRCH.CD.WT", "TM.VAL.MRCH.CD.WT",
    # Innovation & R&D
    "IP.PAT.RESD", "IP.TMK.RSCT", "BM.GSR.ROYL.CD", "BX.GSR.ROYL.CD", "GB.XPD.RSDV.GD.ZS",
    "SP.POP.SCIE.RD.P6", "IC.BUS.NREG", "IC.BUS.NDNS.ZS", "IT.NET.USER.ZS",
    # Military
    "MS.MIL.XPND.CD", "MS.MIL.TOTL.P1",
    # Financial Markets & Monetary
    "CM.MKT.LCAP.CD", "CM.MKT.LCAP.GD.ZS", "CM.MKT.TRAD.CD", "CM.MKT.TRNR", "FS.AST.PRVT.GD.ZS",
    "FM.LBL.BMNY.ZG", "FR.INR.RINR", "FR.INR.LEND", "FP.CPI.TOTL.ZG",
    # Education
    "SE.TER.ENRR", "SE.SEC.ENRL", "SE.PRM.ENRL", "SE.XPD.TOTL.GD.ZS", "SE.TER.CUAT.BA.ZS",
    "SE.SEC.CUAT.UP.ZS",
    # Infrastructure & Investment
    "NE.GDI.FTOT.CD", "NE.GDI.FTOT.ZS", "NE.GDI.TOTL.CD", "NE.GDI.TOTL.ZS", "NY.GNS.ICTR.ZS",
    "NY.GDS.TOTL.ZS", "IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1", "IS.RRS.TOTL.KM", "IS.RRS.GOOD.MT.K6",
    "IS.SHP.GCNW.XQ", "LP.LPI.OVRL.XQ",
    # Geology, Resources, Emissions
    "AG.LND.TOTL.K2", "ER.H2O.INTR.K3", "TX.VAL.MMTL.ZS.UN", "TM.VAL.MMTL.ZS.UN", "AG.PRD.CREL.MT",
    "AG.PRD.FOOD.XD", "NY.GDP.TOTL.RT.ZS", "TX.VAL.FUEL.ZS.UN", "TM.VAL.FUEL.ZS.UN", "TX.VAL.AGRI.ZS.UN",
    "TM.VAL.AGRI.ZS.UN", "TX.VAL.FOOD.ZS.UN", "TM.VAL.FOOD.ZS.UN", "EG.IMP.CONS.ZS",
    # Labor & Inequality
    "SL.UEM.TOTL.ZS", "SI.POV.GINI", "SI.DST.10TH.10",
    # Indebtedness & Fiscal
    "BN.CAB.XOKA.GD.ZS", "GC.DOD.TOTL.GD.ZS", "GC.XPN.TOTL.GD.ZS", "GC.NLD.TOTL.GD.ZS", "GC.XPN.INTP.RV.ZS",
    "GC.XPN.INTP.ZS", "GC.XPN.TRFT.ZS", "DT.DOD.DECT.GN.ZS", "DT.TDS.DECT.EX.ZS"
]
WDI_CODES = sorted(set(WDI_CODES))

def fetch_wb_wdi():
    print(f"Fetching WDI {len(WDI_CODES)} indicators across all economies (full history, single-series)...")
    wb.get_options['timeout'] = 60
    frames = []
    failed = []

    for i, code in enumerate(WDI_CODES, 1):
        success = False
        for attempt in range(1, 4):
            try:
                # Fetch a single indicator across all economies
                d = wb.data.DataFrame(code, time=range(YEAR_FLOOR, 2026), labels=False)
                if d is not None and not d.empty:
                    d = d.reset_index()
                    year_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
                    id_cols = [c for c in d.columns if c not in year_cols]
                    m = pd.melt(d, id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="value").dropna(subset=["value"])
                    m["year"] = m["year"].str.replace("YR", "").astype(int)

                    # Normalize column names
                    if "economy" in m.columns:
                        m = m.rename(columns={"economy": "country_code"})
                    if "series" in m.columns:
                        m = m.rename(columns={"series": "indicator_code"})
                    else:
                        m["indicator_code"] = code

                    frames.append(tidy(m, source="WB-WDI"))
                success = True
                break
            except Exception as e:
                if any(x in str(e) for x in ["Connection failed", "Failed to resolve", "nodename"]):
                    print(f"  Network offline while fetching {code}. Skipping remaining indicators.")
                    break
                time.sleep(attempt)

        if not success:
            print(f"  [{i}/{len(WDI_CODES)}] Failed: {code}")
            failed.append(code)
        elif i % 25 == 0 or i == len(WDI_CODES):
            print(f"  [{i}/{len(WDI_CODES)}] processed ({len(frames)} series ingested)")

    if failed:
        print(f"  Completed with {len(failed)} failed indicators: {failed}")

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)
    return tidy(out, source="WB-WDI")

df_wb_wdi = cached("wb_wdi", fetch_wb_wdi, refresh="wb_wdi" in REFRESH)

# 2b. World Bank - Other Databases (db 1, 3, 40, 58, 63)
- WGI (db 3): FD-09 `.EST` series only (`CC.EST`, `GE.EST`, `PV.EST`, `RL.EST`, `RQ.EST`). Drop `.PER.RNK` and `VA.EST`.
- Doing Business (db 1): 8 referenced series directly.
- UHC (db 58): `SH.UHC.SRVS.CV.XD`.
- HCI (db 63): `HD.HCI.OVRL`, `HD.HCI.EYRS`, `HD.HCI.HLOS`.
- Population projections (db 40, P-4): years 2026–2040 only as `PROJ.*` with `is_projection = True`.


In [ ]:
# 2b. World Bank - Other Databases (C-2.1: single-code requests have no 'series' level; one helper now serves every block)
def _wb_long(d, source, codes, prefix=None, projection=False):
    """Melt a wbgapi wide frame to the master schema. Handles single-code requests (no 'series' index level)."""
    d = d.reset_index()
    y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
    m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
    m["year"] = m["year"].str.replace("YR", "").astype(int)
    m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
    if "indicator_code" not in m.columns:
        m["indicator_code"] = codes[0]
    if prefix:
        m["indicator_code"] = prefix + m["indicator_code"]
    if projection:
        m["is_projection"] = True
    return tidy(m, source=source)

def fetch_wb_other():
    wb.get_options['timeout'] = 30
    frames = []
    blocks = [
        # (source, db, codes, extra kwargs). WGI keeps .EST only (FD-09); PV.EST dropped with its component (K-3).
        ("WB-WGI", 3, ["GOV_WGI_CC.EST", "GOV_WGI_GE.EST", "GOV_WGI_RL.EST", "GOV_WGI_RQ.EST"], {}),
        ("WB-DB", 1, ["IC.CNST.PRMT.DFRN.DB1619", "IC.CNST.PRMT.DFRN.DB0615", "PROT.MINOR.INV.DFRN.DB1519", "PROT.MINOR.INV.DFRN.DB0614",
                      "ENF.CONT.COEN.DB1719.DFRN", "ENF.CONT.COEN.DB0415", "IC.REG.DFRN.PC.DFRN", "IC.CRED.ACC.LGL.RGHT.012.XD.DB1519.DFRN"], {}),
        ("WB-UHC", 58, ["SH.UHC.SRVS.CV.XD"], {}),
        ("WB-HCI", 63, ["HD.HCI.OVRL", "HD.HCI.EYRS", "HD.HCI.HLOS"], {}),
        ("WB-PROJ", 40, ["SP.POP.1564.TO", "SP.POP.DPND"], {"time": range(2026, 2041)}),   # P-4: projections only
    ]
    for source, db, codes, kw in blocks:
        try:
            d = wb.data.DataFrame(codes, db=db, labels=False, **kw)
            if d is None or d.empty:
                print(f"  {source}: no rows returned")
                continue
            frames.append(_wb_long(d, source, codes,
                                   prefix="PROJ." if source == "WB-PROJ" else None, projection=source == "WB-PROJ"))
            if source == "WB-WGI":
                frames[-1]["indicator_code"] = frames[-1]["indicator_code"].str.replace("GOV_WGI_", "")
        except Exception as e:
            print(f"  {source} failed: {e}")
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)
    return tidy(out)

df_wb_other = cached("wb_other", fetch_wb_other, refresh="wb_other" in REFRESH)

# 2c. World Bank EdStats (db 12)
Barro-Lee educational attainment and PISA test scores. Full history.


In [ ]:
# 2c. World Bank EdStats (db 12)
EDSTATS_CODES = [
    "BAR.SCHL.25UP", "BAR.NOED.25UP.ZS", "BAR.PRM.CMPT.25UP.ZS", "BAR.SEC.CMPT.25UP.ZS", "BAR.TER.CMPT.25UP.ZS",
    "LO.PISA.MAT", "LO.PISA.REA", "LO.PISA.SCI"
]

def fetch_wb_edstats():
    wb.get_options['timeout'] = 30
    try:
        d = wb.data.DataFrame(EDSTATS_CODES, db=12, labels=False)
        if d is not None and not d.empty:
            d = d.reset_index()
            y_cols = [c for c in d.columns if re.match(r"^YR\d{4}$", str(c))]
            m = pd.melt(d, id_vars=[c for c in d.columns if c not in y_cols], value_vars=y_cols, var_name="year", value_name="value")
            m["year"] = m["year"].str.replace("YR", "").astype(int)
            m = m.rename(columns={"economy": "country_code", "series": "indicator_code"})
            return tidy(m, source="WB-EDSTATS")
    except Exception as e:
        print(f"WB EdStats fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_wb_edstats = cached("wb_edstats", fetch_wb_edstats, refresh="wb_edstats" in REFRESH)


# 2d. IMF WEO (SDMX 3.0)
Fetches macro indicators and projections via IMF SDMX REST API.
Retention (FD-10): keeps tree-referenced series + `NGDP_RPCH` and `GGXWDN_NGDP`.
Projections (year >= 2026) get `is_projection = True`.


In [ ]:
# 2d. IMF WEO
import io
import pandas as pd

IMF_API = "https://api.imf.org/external/sdmx/3.0/data/dataflow"
IMF_CODES = [
    "GGXWDG_NGDP", "GGXCNL_NGDP", "NGDP_RPCH", "BCA_NGDPD", "PCPIPCH", "LUR",
    "NGDPD", "NGDPDPC", "PPPGDP", "PPPPC", "NGSD_NGDP", "NID_NGDP", "GGX_NGDP", "GGXWDN_NGDP"
]

def fetch_imf_weo():
    rows = []
    print("Fetching IMF WEO via SDMX...")
    for ind in IMF_CODES:
        try:
            q_url = f"{IMF_API}/IMF.RES/WEO/~/*.{ind}.A"
            r = http_get(q_url, headers={"Accept": "text/csv"}, timeout=30)
            if r.status_code == 200 and r.text.strip():
                df_c = pd.read_csv(io.StringIO(r.text))
                geo_col = "COUNTRY" if "COUNTRY" in df_c.columns else ("REF_AREA" if "REF_AREA" in df_c.columns else None)
                if geo_col is None or "OBS_VALUE" not in df_c.columns or "TIME_PERIOD" not in df_c.columns:
                    continue

                df_c = df_c[df_c[geo_col].isin(UNIVERSE)]
                if df_c.empty:
                    continue

                df_c = df_c.dropna(subset=["TIME_PERIOD", "OBS_VALUE"])
                for iso3, p, v in zip(df_c[geo_col], df_c["TIME_PERIOD"], df_c["OBS_VALUE"]):
                    try:
                        y = int(p)
                        val = float(v)
                        rows.append({
                            "country_code": str(iso3).strip(),
                            "indicator_code": "IMF." + ind,
                            "year": y,
                            "value": val,
                            "source": "IMF-WEO",
                            "is_projection": (y >= PROJ_FROM)
                        })
                    except (ValueError, TypeError):
                        continue
        except Exception as e:
            print(f"  IMF series {ind} failed: {e}")
            if any(err in str(e) for err in ("Connection failed", "Failed to resolve", "nodename")):
                print("  Network offline: skipping remaining IMF series.")
                break

    return tidy(pd.DataFrame(rows))

df_imf_weo = cached("imf_weo", fetch_imf_weo, refresh="imf_weo" in REFRESH)

# 2e. IMF COFER (Reserve Currency)
Currency shares of world allocated reserves.
FD-07 / D12: EUR share assigned to nobody (euro-area members receive NO reserve currency rows).
CHF share goes to CHE and LIE. Other currencies unchanged. Non-issuers keep structural 0.


In [ ]:
# 2e. IMF COFER (Reserve Currency)
from collections import defaultdict

EURO_AREA = {
    "AUT", "BEL", "BGR", "HRV", "CYP", "EST", "FIN", "FRA", "DEU", "GRC", "IRL", "ITA",
    "LVA", "LTU", "LUX", "MLT", "NLD", "PRT", "SVK", "SVN", "ESP"
}
COFER_ISSUERS = {
    "USD": ["USA"],
    "EUR": [],             # D12: EUR share assigned to nobody; members receive no rows
    "JPY": ["JPN"],
    "GBP": ["GBR"],
    "CNY": ["CHN"],
    "AUD": ["AUS"],
    "CAD": ["CAN"],
    "CHF": ["CHE", "LIE"]
}

def fetch_imf_cofer():
    try:
        raw = None
        for key in ("G001.AFXRA.*.SHRO_PT.Q", "G001.*.*.*.Q"):
            try:
                r = http_get(f"{IMF_API}/IMF.STA/COFER/~/{key}", headers={"Accept": "text/csv"}, timeout=60)
                if r.status_code == 200 and r.text.strip():
                    raw = pd.read_csv(io.StringIO(r.text))
                    break
            except Exception:
                continue

        if raw is None or raw.empty:
            print("COFER fetch failed: no data returned from IMF SDMX API.")
            return pd.DataFrame(columns=STD_COLS)

        cols = {c.upper(): c for c in raw.columns}
        ccy_c = next((cols[c] for c in cols if "CURRENCY" in c), None)
        tr_c = next((cols[c] for c in cols if "TRANSFORMATION" in c), None)
        ind_c = cols.get("INDICATOR")

        if ccy_c is None or "OBS_VALUE" not in raw.columns or "TIME_PERIOD" not in raw.columns:
            return pd.DataFrame(columns=STD_COLS)

        d = raw.copy()
        if tr_c is not None:
            d = d[d[tr_c].astype(str).str.contains("SHR", case=False)]
        if ind_c is not None and "AFXRA" in set(d[ind_c].astype(str)):
            d = d[d[ind_c].astype(str) == "AFXRA"]

        d["ccy"] = d[ccy_c].astype(str).str.replace("CI_", "", regex=False).str.upper()
        d = d[d["ccy"].isin(COFER_ISSUERS)]
        d["year"] = pd.to_numeric(d["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
        d["value"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
        d = d.dropna(subset=["year", "value"]).sort_values("TIME_PERIOD").groupby(["ccy", "year"], as_index=False).last()

        currency_shares = defaultdict(dict)
        all_years = set()
        for _, r in d.iterrows():
            ccy = r["ccy"]
            y = int(r["year"])
            currency_shares[ccy][y] = float(r["value"])
            all_years.add(y)

        rows = []
        for y in sorted(all_years):
            issuers_for_year = set()
            for ccy, cty_list in COFER_ISSUERS.items():
                share = currency_shares[ccy].get(y, 0.0)
                for iso3 in cty_list:
                    if iso3 in UNIVERSE:
                        rows.append({
                            "country_code": iso3,
                            "indicator_code": "IMF.COFER_SHARE",
                            "year": y,
                            "value": share,
                            "source": "IMF-COFER",
                            "is_projection": False
                        })
                        issuers_for_year.add(iso3)

            # Structural zero for non-issuers, excluding euro area (FRD D12)
            for c in UNIVERSE:
                if c not in EURO_AREA and c not in issuers_for_year:
                    rows.append({
                        "country_code": c,
                        "indicator_code": "IMF.COFER_SHARE",
                        "year": y,
                        "value": 0.0,
                        "source": "IMF-COFER",
                        "is_projection": False
                    })

        return tidy(pd.DataFrame(rows))
    except Exception as e:
        print(f"COFER fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)

df_imf_cofer = cached("imf_cofer", fetch_imf_cofer, refresh="imf_cofer" in REFRESH)

# 2f. US EIA International
P-7 Secrets: API key read from `EIA_API_KEY` environment variable.
P-4 Efficient routes: Fixed product list: 44 (total primary energy) and 2 (electricity).
Exact codes: `EIA.ENERGY.PRODUCTION_QBTU`, `EIA.ENERGY.SELF_SUFF`, `EIA.ELECTRICITY.GENERATION.BKWH`.


In [ ]:
# 2f. US EIA International (C-2.7: paged requests, rows filtered by activity and unit, consumption added for self-sufficiency)
EIA_PRODUCTS = {                                   # productId -> ({activity-name keyword: indicator code}, unit keywords)
    44: ({"production": "EIA.ENERGY.PRODUCTION_QBTU", "consumption": "EIA.ENERGY.CONSUMPTION_QBTU"}, ("btu",)),
    2:  ({"generation": "EIA.ELECTRICITY.GENERATION.BKWH"}, ("kwh", "kilowatt")),
}

def fetch_eia():
    if not EIA_API_KEY:
        print("EIA_API_KEY not configured. Skipping live EIA fetch.")
        return pd.DataFrame(columns=STD_COLS)
    base_url = "https://api.eia.gov/v2/international/data/"
    rows, seen = [], []
    for pid, (activities, unit_kws) in EIA_PRODUCTS.items():
        offset = 0
        while True:                                # the API returns at most 5000 rows per request
            params = {"api_key": EIA_API_KEY, "frequency": "annual", "data[0]": "value", "facets[productId][]": pid,
                      "length": 5000, "offset": offset, "sort[0][column]": "period", "sort[0][direction]": "asc"}
            r = http_get(base_url, params=params, timeout=60)
            if r.status_code != 200:
                print(f"  EIA product {pid}: HTTP {r.status_code} at offset {offset}")
                break
            data = r.json().get("response", {}).get("data", [])
            for row in data:
                act = str(row.get("activityName", "")).lower()
                unit = str(row.get("unit", "")).lower()
                seen.append((pid, row.get("activityName"), row.get("unit")))
                code = next((c for k, c in activities.items() if k in act), None)
                if code is None or not any(k in unit for k in unit_kws):
                    continue
                iso3 = row.get("countryRegionId") or to_iso3([row.get("countryRegionName")]).iloc[0]
                try:
                    rows.append({"country_code": iso3, "indicator_code": code, "year": int(row["period"]), "value": float(row["value"]),
                                 "source": "EIA", "is_projection": False})
                except (ValueError, TypeError, KeyError):
                    continue
            if len(data) < 5000:
                break
            offset += 5000
    print("  EIA (product, activity, unit) combinations seen:")
    print(pd.Series(seen).value_counts().head(12).to_string())      # lets you confirm the filters picked the right rows
    return tidy(pd.DataFrame(rows))

df_eia = cached("eia", fetch_eia, refresh="eia" in REFRESH)

# 2g. V-Dem (v16)
Varieties of Democracy v16 country-year panel.
Retention (FD-10): keeps only referenced governance and conflict series.


In [ ]:
# 2g. V-Dem
VDEM_URL = "https://www.v-dem.net/media/datasets/V-Dem-CY-FullOthers-v16_csv.zip"
VDEM_KEEP = [
    "v2cltrnslw", "v2excrptps", "v2exthftps", "v2clrspct", "v2x_rule", "v2cacamps",
    "v2x_corr", "v2x_pubcorr", "v2x_execorr", "v2xcl_prpty", "v2x_jucon", "v2xeg_eqdr",
    "v2cagenmob", "v2cademmob", "v2x_clphy", "v2smpolsoc"
]

def fetch_vdem():
    print("Streaming V-Dem v16 dataset into memory...")
    try:
        r = http_get(VDEM_URL, timeout=120)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            # Read only relevant columns
            usecols = ["country_text_id", "year"] + VDEM_KEEP
            d = pd.read_csv(z.open(csv_fn), usecols=lambda c: c in usecols)
        d = d.rename(columns={"country_text_id": "country_code"})
        m = pd.melt(d, id_vars=["country_code", "year"], value_vars=[c for c in VDEM_KEEP if c in d.columns], var_name="indicator_code", value_name="value")
        m["indicator_code"] = "VDEM." + m["indicator_code"]
        return tidy(m, source="VDEM")
    except Exception as e:
        print(f"V-Dem fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)

df_vdem = cached("vdem", fetch_vdem, refresh="vdem" in REFRESH)


# 2h. ND-GAIN (2026)
University of Notre Dame Global Adaptation Initiative index.
Retention (D18): `NDGAIN.VULNERABILITY` and `NDGAIN.READINESS` are wired into `acts_of_nature`.
All remaining subcomponents are kept with status `pdf_listed_unwired`.
`NDGAIN.GAIN` composite is dropped.


In [ ]:
# 2h. ND-GAIN
NDGAIN_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"

def fetch_ndgain():
    print("Streaming ND-GAIN 2026 archive into memory...")
    try:
        r = http_get(NDGAIN_URL, timeout=90)
        r.raise_for_status()
        rows = []
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            for fn in z.namelist():
                if not fn.endswith(".csv"):
                    continue
                # Determine metric code from filename
                stem = Path(fn).stem.lower()
                if "vulnerability" in stem:
                    code = "NDGAIN.VULNERABILITY"
                elif "readiness" in stem:
                    code = "NDGAIN.READINESS"
                elif "gain" in stem:
                    continue  # drop composite per FRD §9.4
                else:
                    code = f"NDGAIN.{stem.upper()}"
                
                try:
                    df_c = pd.read_csv(z.open(fn))
                    # Standard NDGAIN csv has ISO3 in first column, years in remaining
                    iso_col = df_c.columns[0]
                    y_cols = [c for c in df_c.columns if re.match(r"^\d{4}$", str(c))]
                    m = pd.melt(df_c, id_vars=[iso_col], value_vars=y_cols, var_name="year", value_name="value")
                    m = m.rename(columns={iso_col: "country_code"})
                    m["indicator_code"] = code
                    rows.append(m)
                except Exception:
                    continue
        if rows:
            out = pd.concat(rows, ignore_index=True)
            return tidy(out, source="ND-GAIN")
    except Exception as e:
        print(f"ND-GAIN fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_ndgain = cached("ndgain", fetch_ndgain, refresh="ndgain" in REFRESH)


# 2i. Our World in Data (OWID)
FD-01: Verifies the `Code` column! Never assigns a row to a country the `Code` column does not name.
FD-05: COVID excess mortality: mean P-score over 2020-01-01 to 2022-12-31, dated at 2022.
FD-06: WID inequality series (`WID.TOP1_INCOME`, `WID.TOP10_INCOME`, `WID.TOP1_WEALTH`, `WID.TOP10_WEALTH`).
FAS nuclear warhead stockpiles + structural zeros for non-nuclear states.
Energy, TI corruption perception index, conflict deaths.


In [ ]:
# 2i. Our World in Data
OWID_CHARTS = {
    "nuclear-warhead-stockpiles": "FAS.NUCLEAR_STOCKPILE",
    "happiness-cantril-ladder": "WHR.CANTRIL_LADDER",
    "electricity-generation": "OWID.ELECTRICITY_GEN_TWH",
    "primary-energy-consumption": "OWID.PRIMARY_ENERGY_TWH",
    "oil-production-by-country": "OWID.OIL_PRODUCTION_TWH",
    "gas-production-by-country": "OWID.GAS_PRODUCTION_TWH",
    "coal-production-by-country": "OWID.COAL_PRODUCTION_TWH",
    "ti-corruption-perception-index": "TI.CPI",
    "deaths-in-armed-conflicts": "UCDP.INTERNAL_CONFLICT_DEATHS",
    "share-of-the-top-1-in-pre-tax-national-income": "WID.TOP1_INCOME",
    "share-of-the-top-10-in-pre-tax-national-income-pip": "WID.TOP10_INCOME",
    "wealth-share-richest-one-percent": "WID.TOP1_WEALTH",
    "wealth-share-richest-ten-percent": "WID.TOP10_WEALTH",
    "excess-mortality-p-scores-average-baseline": "COVID.EXCESS_MORTALITY_PSCORE"
}

def fetch_owid():
    print(f"Fetching {len(OWID_CHARTS)} OWID charts with strict Code-column checking...")
    rows = []
    base_url = "https://ourworldindata.org/grapher/{slug}.csv"
    
    for slug, code in OWID_CHARTS.items():
        try:
            url = base_url.format(slug=slug)
            r = http_get(url, timeout=30)
            if r.status_code != 200:
                print(f"  OWID chart {slug} HTTP {r.status_code}")
                continue
            df_c = pd.read_csv(io.StringIO(r.text))
            # FD-01: check Code column
            if "Code" not in df_c.columns:
                continue
            df_c = df_c.dropna(subset=["Code"]).copy()
            df_c = df_c[df_c["Code"].isin(UNIVERSE)]
            
            # Value column is the first non-entity/code/year/day column
            meta_cols = {"Entity", "Code", "Year", "Day"}
            val_cols = [c for c in df_c.columns if c not in meta_cols]
            if not val_cols:
                continue
            val_col = val_cols[0]
            
            # High-frequency date handling (FD-05 COVID excess mortality)
            if "Day" in df_c.columns and code == "COVID.EXCESS_MORTALITY_PSCORE":
                df_c["Day"] = pd.to_datetime(df_c["Day"], errors="coerce")
                covid_window = df_c[(df_c["Day"] >= "2020-01-01") & (df_c["Day"] <= "2022-12-31")]
                agg = covid_window.groupby("Code")[val_col].mean().reset_index()
                agg["Year"] = 2022
                df_c = agg
            elif "Day" in df_c.columns:
                df_c["Year"] = pd.to_datetime(df_c["Day"], errors="coerce").dt.year
                df_c = df_c.groupby(["Code", "Year"])[val_col].mean().reset_index()

            # Nuclear warheads: structural zeros for non-reporting states
            if code == "FAS.NUCLEAR_STOCKPILE":
                nuclear_states = set(df_c["Code"].unique())
                years = df_c["Year"].dropna().unique()
                zero_rows = []
                for y in years:
                    for c in UNIVERSE:
                        if c not in nuclear_states:
                            zero_rows.append({"Code": c, "Year": y, val_col: 0.0})
                if zero_rows:
                    df_c = pd.concat([df_c, pd.DataFrame(zero_rows)], ignore_index=True)

            for _, row in df_c.iterrows():
                rows.append({
                    "country_code": row["Code"],
                    "indicator_code": code,
                    "year": int(row["Year"]),
                    "value": float(row[val_col]),
                    "source": "OWID",
                    "is_projection": False
                })
        except Exception as e:
            print(f"  OWID chart {slug} failed: {e}")
            if "Connection failed" in str(e) or "Failed to resolve" in str(e) or "nodename" in str(e):
                print("  Network offline: skipping remaining OWID charts.")
                break
            
    out = pd.DataFrame(rows)
    if not out.empty:
        # FD-01 Assert: every emitted row has country_code == Code in UNIVERSE
        assert out["country_code"].isin(UNIVERSE).all()
    return tidy(out)

df_owid = cached("owid", fetch_owid, refresh="owid" in REFRESH)


# 2j. ILOSTAT
FD-08: Uses bulk download files as primary route.
Male weekly hours (`HOW_2EMP_SEX_NB_A`) -> `ILO.HOURS_WEEKLY_MODELLED_MALE` (exact).
Total weekly hours fallback -> `ILO.HOURS_WEEKLY_MODELLED` (substitute).
Male LFPR (`EAP_2WAP_SEX_AGE_RT_A`) -> `ILO.LFPR_MODELLED_MALE`.
Long-term unemployment share (`UNE_TUNE_SEX_AGE_DUR_NB_A`) -> `ILO.UNEMP_LONGTERM_SHARE`.
Strikes days lost and dispute counts. Collective bargaining coverage.


In [ ]:
# 2j. ILOSTAT
import io
import pandas as pd

ILO_URL = "https://rplumber.ilo.org/data/indicator/?id={ind}&timefrom=2000&format=.csv"


def load_ilo_df(url):
    r = http_get(url, timeout=60)
    if r.status_code == 200 and r.text.strip():
        try:
            df = pd.read_csv(io.StringIO(r.text))
        except Exception:
            df = pd.read_csv(io.BytesIO(r.content))
        df.columns = [str(c).lower() for c in df.columns]
        return df
    return None


def fetch_ilostat():
    print("Fetching ILOSTAT indicators...")
    rows = []

    # 1. Weekly hours (male & total)
    try:
        df_h = load_ilo_df(ILO_URL.format(ind="HOW_2EMP_SEX_NB_A"))
        if df_h is not None:
            for sex_code, ind_name in [("SEX_M", "ILO.HOURS_WEEKLY_MODELLED_MALE"), ("SEX_T", "ILO.HOURS_WEEKLY_MODELLED")]:
                sub = df_h[df_h["sex"].astype(str).str.upper() == sex_code].copy()
                sub["country_code"] = to_iso3(sub["ref_area"])
                sub = sub.dropna(subset=["country_code", "time", "obs_value"])
                for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                    rows.append({"country_code": c, "indicator_code": ind_name, "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
    except Exception as e:
        print(f"  ILO hours failed: {e}")

    # 2. Male LFPR
    try:
        df_lf = load_ilo_df(ILO_URL.format(ind="EAP_2WAP_SEX_AGE_RT_A"))
        if df_lf is not None:
            sub = df_lf[(df_lf["sex"].astype(str).str.upper() == "SEX_M") & (df_lf["classif1"].astype(str).str.contains("GE15", na=False))].copy()
            sub["country_code"] = to_iso3(sub["ref_area"])
            sub = sub.dropna(subset=["country_code", "time", "obs_value"])
            for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                rows.append({"country_code": c, "indicator_code": "ILO.LFPR_MODELLED_MALE", "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
    except Exception as e:
        print(f"  ILO LFPR failed: {e}")

    # 3. Unemployment Rate, Strikes & Collective Bargaining
    indicators = [
        ("UNE_2EAP_SEX_AGE_RT_A", "ILO.UNEMP_RATE_MODELLED"),
        ("STR_DAYS_ECO_NB_A",     "ILO.STRIKES_DAYS_NOT_WORKED"),
        ("STR_TSTR_ECO_NB_A",     "ILO.STRIKES_COUNT"),
        ("ILR_CBCT_NOC_RT_A",     "ILO.COLLECTIVE_BARGAINING_COVERAGE"),
    ]

    for ind_id, code_name in indicators:
        try:
            df_ind = load_ilo_df(ILO_URL.format(ind=ind_id))
            if df_ind is not None:
                # Keep total sex if dimension exists
                if "sex" in df_ind.columns:
                    df_ind = df_ind[df_ind["sex"].astype(str).str.upper().isin(["SEX_T", "TOTAL", "T"])]
                # Keep total/aggregate breakdowns (avoids row duplication by age or ISIC economic sector)
                if "classif1" in df_ind.columns:
                    c1_ok = (
                        df_ind["classif1"].isna() |
                        df_ind["classif1"].astype(str).str.endswith("_TOTAL") |
                        df_ind["classif1"].astype(str).str.contains("GE15", na=False)
                    )
                    df_ind = df_ind[c1_ok]

                df_ind["country_code"] = to_iso3(df_ind["ref_area"])
                sub = df_ind.dropna(subset=["country_code", "time", "obs_value"])
                for c, y, v in zip(sub["country_code"], sub["time"], sub["obs_value"]):
                    rows.append({"country_code": c, "indicator_code": code_name, "year": int(float(y)), "value": float(v), "source": "ILO", "is_projection": False})
        except Exception as e:
            print(f"  ILO {ind_id} failed: {e}")

    return tidy(pd.DataFrame(rows))

df_ilostat = cached("ilostat", fetch_ilostat, refresh="ilostat" in REFRESH)

# 2k. BIS Bulk Files
Credit to the non-financial sector (`WS_TC`): `TC.G` (government), `TC.P` (private), `TC.N` (total).
Central bank policy rates (`WS_CBPOL`): end of period and average.
Probes for debt securities and cross-border claims.


In [ ]:
# 2k. BIS Bulk Files
BIS_BULK_URLS = {
    "WS_TC": "https://data.bis.org/static/bulk/WS_TC_csv_flat.zip",
    "WS_CBPOL": "https://data.bis.org/static/bulk/WS_CBPOL_csv_flat.zip"
}


def clean_bis_df(df):
    """Normalize BIS SDMX 'CODE:Description' column headers and cell values to CODE."""
    df.columns = [str(c).split(":")[0].strip() for c in df.columns]
    if "OBS_VALUE" in df.columns:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
    for c in df.columns:
        if c not in ("TIME_PERIOD", "OBS_VALUE") and not pd.api.types.is_numeric_dtype(df[c]):
            if c in ("UNIT_TYPE", "UNIT_MEASURE"):
                df[c + "_LABEL"] = df[c].astype(str)
            df[c] = df[c].astype(str).str.split(":").str[0].str.strip()
    return df


def fetch_bis():
    rows = []
    print("Fetching BIS bulk files...")

    # 1. Total credit (WS_TC)
    try:
        r = http_get(BIS_BULK_URLS["WS_TC"], timeout=90)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            df_tc = pd.read_csv(z.open(csv_fn), low_memory=False)
        df_tc = clean_bis_df(df_tc)

        # Keep % of GDP (UNIT_TYPE == '770' or 'P'), all lenders ('A'), and break-adjusted ('A')
        if "UNIT_TYPE" in df_tc.columns:
            mask = df_tc["UNIT_TYPE"].isin(["770", "P"])
            if "UNIT_TYPE_LABEL" in df_tc.columns:
                mask = mask | df_tc["UNIT_TYPE_LABEL"].str.contains("GDP", case=False, na=False)
            df_tc = df_tc[mask].copy()

        if "TC_LENDERS" in df_tc.columns and (df_tc["TC_LENDERS"] == "A").any():
            df_tc = df_tc[df_tc["TC_LENDERS"] == "A"].copy()

        if "TC_ADJUST" in df_tc.columns:
            if (df_tc["TC_ADJUST"] == "A").any():
                df_tc = df_tc[df_tc["TC_ADJUST"] == "A"].copy()
            elif (df_tc["TC_ADJUST"] == "N").any():
                df_tc = df_tc[df_tc["TC_ADJUST"] == "N"].copy()

        # Prefer market valuation ('M') over nominal ('N') to eliminate duplicates
        cty_col = "BORROWERS_CTY" if "BORROWERS_CTY" in df_tc.columns else "REF_AREA"
        if "VALUATION" in df_tc.columns:
            df_tc["_rk"] = df_tc["VALUATION"].map({"M": 0, "N": 1}).fillna(2)
            df_tc = df_tc.sort_values("_rk").drop_duplicates([cty_col, "TC_BORROWERS", "TIME_PERIOD"]).drop(columns=["_rk"])

        for sector, code_name in [("G", "BIS.TC.G.PCT_GDP"), ("P", "BIS.TC.P.PCT_GDP"), ("N", "BIS.TC.N.PCT_GDP")]:
            sub = df_tc[df_tc["TC_BORROWERS"] == sector].copy()
            if sub.empty:
                continue
            sub = sub.reset_index(drop=True)
            iso = to_iso3(sub[cty_col], src="ISO2")
            sub["country_code"] = iso.values if hasattr(iso, "values") else iso
            sub["year"] = pd.to_numeric(sub["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
            sub = sub.dropna(subset=["country_code", "year", "OBS_VALUE"])
            sub["year"] = sub["year"].astype(int)

            # Take annual mean
            agg = sub.groupby(["country_code", "year"])["OBS_VALUE"].mean().reset_index()
            for _, r_val in agg.iterrows():
                rows.append({
                    "country_code": r_val["country_code"],
                    "indicator_code": code_name,
                    "year": int(r_val["year"]),
                    "value": float(r_val["OBS_VALUE"]),
                    "source": "BIS",
                    "is_projection": False
                })
    except Exception as e:
        print(f"  BIS WS_TC failed: {e}")

    # 2. Policy rates (WS_CBPOL)
    try:
        r = http_get(BIS_BULK_URLS["WS_CBPOL"], timeout=90)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_fn = [f for f in z.namelist() if f.endswith(".csv")][0]
            df_cb = pd.read_csv(z.open(csv_fn), low_memory=False)
        df_cb = clean_bis_df(df_cb)

        ref_col = "REF_AREA" if "REF_AREA" in df_cb.columns else "BORROWERS_CTY"
        df_cb = df_cb.reset_index(drop=True)
        iso = to_iso3(df_cb[ref_col], src="ISO2")
        df_cb["country_code"] = iso.values if hasattr(iso, "values") else iso
        df_cb["year"] = pd.to_numeric(df_cb["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
        df_cb = df_cb.dropna(subset=["country_code", "year", "OBS_VALUE"])
        df_cb["year"] = df_cb["year"].astype(int)

        df_cb = df_cb.sort_values("TIME_PERIOD")
        agg = df_cb.groupby(["country_code", "year"])["OBS_VALUE"].agg(end="last", avg="mean").reset_index()      # C-2.8: year-end and annual average
        for code_name, col in [("BIS.CBPOL.POLICY_RATE_END", "end"), ("BIS.CBPOL.POLICY_RATE_AVG", "avg")]:
            for _, r_val in agg.iterrows():
                rows.append({"country_code": r_val["country_code"], "indicator_code": code_name, "year": int(r_val["year"]),
                             "value": float(r_val[col]), "source": "BIS", "is_projection": False})
    except Exception as e:
        print(f"  BIS WS_CBPOL failed: {e}")

    return tidy(pd.DataFrame(rows))


df_bis = cached("bis", fetch_bis, refresh="bis" in REFRESH)

# 2l. World Happiness Report (WHR)
FD-04: Probes programmatic routes in order for Gallup World Poll indicators:
Life Ladder, Social Support, Positive Affect.


In [ ]:
# 2l. World Happiness Report (WHR)
WHR_URLS = [
    "https://files.worldhappiness.report/WHR26_Data_Figure_2.1.xlsx",
    "https://files.worldhappiness.report/WHR25_Data_Figure_2.1v3.xlsx",
    "https://files.worldhappiness.report/WHR24_Data_Figure_2.1.xlsx",
    "https://happiness-report.s3.amazonaws.com/2024/DataForFigure2.1+with+sub+bars+2024.xls",
    "https://worldhappiness.report/ed/2024/DataForFigure2.1.xlsx",
    "https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/World%20Happiness%20Report/World%20Happiness%20Report.csv"
]

def fetch_whr():
    print("Probing WHR programmatic routes (FD-04)...")
    for url in WHR_URLS:
        try:
            r = http_get(url, timeout=40)
            if r.status_code != 200:
                continue

            # Parse as CSV or multi-sheet Excel
            dfs = []
            if url.endswith(".csv"):
                dfs = [pd.read_csv(io.BytesIO(r.content))]
            else:
                try:
                    xl = pd.ExcelFile(io.BytesIO(r.content))
                    dfs = [xl.parse(sheet) for sheet in xl.sheet_names]
                except Exception:
                    dfs = [pd.read_csv(io.BytesIO(r.content))]

            rows = []
            for df_w in dfs:
                # Support both WHR ("Country name") and OWID ("Entity")
                col_c = next((c for c in df_w.columns if any(k in str(c).lower() for k in ["country", "entity"])), None)
                if not col_c:
                    continue

                col_y = next((c for c in df_w.columns if str(c).lower().strip() == "year"), None)
                df_w["country_code"] = to_iso3(df_w[col_c])
                df_w = df_w.dropna(subset=["country_code"])

                for col in df_w.columns:
                    col_l = str(col).lower()
                    if col in [col_c, col_y, "country_code"] or any(ex in col_l for ex in ["explained", "whisker", "dystopia", "rank"]):
                        continue

                    if "ladder" in col_l or "happiness" in col_l or "life evaluation" in col_l:
                        code = "WHR.LIFE_LADDER"
                    elif "social support" in col_l:
                        code = "WHR.SOCIAL_SUPPORT"
                    elif "positive affect" in col_l:
                        code = "WHR.POSITIVE_AFFECT"
                    else:
                        continue

                    # Coerce values safely
                    sub = df_w.dropna(subset=["country_code", col]).copy()
                    sub[col] = pd.to_numeric(sub[col], errors="coerce")
                    sub = sub.dropna(subset=[col])

                    for _, r_val in sub.iterrows():
                        y = int(r_val[col_y]) if col_y and pd.notna(r_val.get(col_y)) else 2024
                        rows.append({
                            "country_code": r_val["country_code"],
                            "indicator_code": code,
                            "year": y,
                            "value": float(r_val[col]),
                            "source": "WHR",
                            "is_projection": False
                        })

                if rows:
                    print(f"  WHR succeeded via {url}")
                    return tidy(pd.DataFrame(rows))

        except Exception:
            continue

    # Fallback to OWID Cantril ladder if direct file unreachable
    print("  Direct WHR excel route unavailable. Documenting social support / positive affect as GAP in ledger.")
    return pd.DataFrame(columns=STD_COLS)

df_whr = cached("whr", fetch_whr, refresh="whr" in REFRESH)

# 2m. UNHCR Displacement
UNHCR population statistics: refugees + asylum seekers + IDPs of origin.
All economies pull -> `UNHCR.DISPLACED_TOTAL`. Zero-filled for non-reporting states.


In [ ]:
# 2m. UNHCR Displacement
UNHCR_API = "https://api.unhcr.org/population/v1/population/"


def fetch_unhcr():
  print("Fetching UNHCR displacement by country of origin...")
  rows = []

  def _to_float(v):
    try:
      return float(v) if v is not None else 0.0
    except (ValueError, TypeError):
      return 0.0

  try:
    page = 1
    while True:
      params = {
          "coo_all": "true",
          "cf_type": "ISO",
          "yearFrom": YEAR_FLOOR,
          "yearTo": 2025,
          "limit": 1000,
          "page": page,
      }
      r = http_get(UNHCR_API, params=params, timeout=60)
      if r.status_code != 200:
        print(f"UNHCR fetch HTTP error: {r.status_code}")
        break

      js = r.json()
      items = js.get("items", [])
      if not items:
        break

      for item in items:
        iso3 = item.get("coo_iso")
        if iso3 not in UNIVERSE:
          continue
        try:
          y = int(item.get("year"))
        except (ValueError, TypeError):
          continue

        # Sum reported categories for this origin record
        val = (
            _to_float(item.get("refugees"))
            + _to_float(item.get("asylum_seekers"))
            + _to_float(item.get("idps"))
        )

        rows.append({
            "country_code": iso3,
            "indicator_code": "UNHCR.DISPLACED_TOTAL",
            "year": y,
            "value": val,
            "source": "UNHCR",
            "is_projection": False,
        })

      max_pages = int(js.get("maxPages", 1) or 1)
      if page >= max_pages:
        break
      page += 1

    out = pd.DataFrame(rows)
    if not out.empty:
      # Aggregate strictly reported figures per country-year
      agg = out.groupby(["country_code", "year"])["value"].sum().reset_index()
      agg["indicator_code"] = "UNHCR.DISPLACED_TOTAL"
      agg["source"] = "UNHCR"
      agg["is_projection"] = False
      return tidy(agg)

  except Exception as e:
    print(f"UNHCR fetch failed: {e}")

  return pd.DataFrame(columns=STD_COLS)


df_unhcr = cached("unhcr", fetch_unhcr, refresh="unhcr" in REFRESH)

# 2n. World Bank PIP
FD-02: Single `country=all` request with `fill_gaps=false`.
Emits `PIP.GINI`, `PIP.DECILE1`, `PIP.DECILE10`. Registered as `substitute_source`.


In [ ]:
# 2n. World Bank PIP
PIP_URL = "https://api.worldbank.org/pip/v1/pip"

def fetch_pip():
    print("Fetching World Bank PIP (FD-02)...")
    try:
        r = http_get(PIP_URL, params={"country": "all", "year": "all", "fill_gaps": "false", "format": "csv"}, timeout=90)
        if r.status_code == 200 and len(r.text) > 100:
            df_pip = pd.read_csv(io.StringIO(r.text))
            rows = []
            for col, ind_code in [("gini", "PIP.GINI"), ("decile1", "PIP.DECILE1"), ("decile10", "PIP.DECILE10")]:
                if col in df_pip.columns:
                    sub = df_pip.dropna(subset=["country_code", "reporting_year", col]).copy()
                    sub["country_code"] = to_iso3(sub["country_code"])
                    for _, r_val in sub.iterrows():
                        rows.append({
                            "country_code": r_val["country_code"],
                            "indicator_code": ind_code,
                            "year": int(r_val["reporting_year"]),
                            "value": float(r_val[col]),
                            "source": "WB-PIP",
                            "is_projection": False
                        })
            return tidy(pd.DataFrame(rows))
    except Exception as e:
        print(f"PIP fetch failed: {e}")
    return pd.DataFrame(columns=STD_COLS)

df_pip = cached("pip", fetch_pip, refresh="pip" in REFRESH)


# 2o. OECD National Accounts at a Glance
Retention (FD-10): keeps only `OECD.HH_SAVINGS_RATE` (household net saving rate). Drops the other 24 series.


In [ ]:
# 2o. OECD NAAG - net household saving rate (National Accounts at a Glance, chapter 5 "Households")
# Key: FREQ.REF_AREA.MEASURE.UNIT_MEASURE.CHAPTER; empty segment = wildcard.
OECD_BASE = "https://sdmx.oecd.org/public/rest/data/"
OECD_FLOWS = ["OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V,1.0", "OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V"]
OECD_MEASURE = "B8NS1M"       # net household saving
OECD_UNIT = "PT_B6N_S1M"      # as % of net household disposable income

def fetch_oecd():
    import io
    print("Fetching OECD household savings rate...")
    key = f"A..{OECD_MEASURE}.{OECD_UNIT}."   # all countries, annual, one series
    need = {"REF_AREA", "MEASURE", "UNIT_MEASURE", "TIME_PERIOD", "OBS_VALUE"}

    for flow in OECD_FLOWS:
        try:
            r = http_get(
                f"{OECD_BASE}{flow}/{key}",
                params={"startPeriod": "2000", "format": "csvfile"},
                timeout=90,
                tries=2,
            )
        except Exception as e:
            print(f"OECD fetch failed ({flow}): {e}")
            continue

        if r.status_code != 200 or not r.text.strip():
            print(f"OECD fetch returned HTTP {r.status_code} / empty body ({flow})")
            continue

        d = pd.read_csv(io.StringIO(r.text))
        if not need <= set(d.columns):
            print(f"OECD response missing columns {need - set(d.columns)} ({flow})")
            continue

        d = d[
            (d["MEASURE"] == OECD_MEASURE)
            & (d["UNIT_MEASURE"] == OECD_UNIT)
            & (d["REF_AREA"].isin(UNIVERSE))
        ].copy()

        d["year"] = pd.to_numeric(d["TIME_PERIOD"], errors="coerce")
        d["value"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
        d = d.dropna(subset=["year", "value"])   # drop unusable rows; nothing is filled
        if d.empty:
            print(f"OECD response had no usable rows in UNIVERSE ({flow})")
            continue

        out = pd.DataFrame({
            "country_code": d["REF_AREA"].values,
            "indicator_code": "OECD.HH_SAVINGS_RATE",
            "year": d["year"].astype(int).values,
            "value": d["value"].astype(float).values,
            "source": "OECD",
            "is_projection": False,
        })
        print(f"OECD: {len(out)} rows, {out['country_code'].nunique()} countries (flow: {flow})")
        return tidy(out)

    return pd.DataFrame(columns=STD_COLS)

df_oecd = cached("oecd", fetch_oecd, refresh="oecd" in REFRESH)

# 2p. ARWU (Shanghai Ranking)
FD-03: Explicit region aliases (Chinese Mainland -> CHN, Taiwan Region -> TWN, Hong Kong SAR -> HKG, Macau SAR -> MAC).
Parses rank bands by leading integer. Zero-fills non-ranking economies for years the ranking exists.
Emits `ARWU.TOP100_COUNT` and `ARWU.TOP500_COUNT`.


In [ ]:
# 2p. ARWU (Shanghai Ranking)
ARWU_API = "https://www.shanghairanking.com/api/pub/v1/arwu/rank"
ARWU_YEARS = range(2019, 2027)
ARWU_REGION_ALIAS = {
    "Chinese Mainland": "CHN", "Mainland China": "CHN", "Taiwan Region": "TWN",
    "Hong Kong SAR": "HKG", "Macau SAR": "MAC", "Hong Kong": "HKG"
}

def fetch_arwu():
    print("Fetching ARWU rankings (FD-03)...")
    frames = []
    for y in ARWU_YEARS:
        try:
            r = http_get(ARWU_API, params={"version": y}, timeout=30)
            if r.status_code != 200:
                continue
            js = r.json()
            data = js.get("data", js) if isinstance(js, dict) else js
            if isinstance(data, dict):
                data = data.get("rankings", data.get("data", []))
            rows = data if isinstance(data, list) else []
            if not rows:
                continue
            
            parsed = []
            for r_entry in rows:
                if not isinstance(r_entry, dict):
                    continue
                rk_raw = str(r_entry.get("ranking", r_entry.get("rank", "")))
                m = re.match(r"^\s*(\d+)", rk_raw)
                rk = int(m.group(1)) if m else 9999
                rg = r_entry.get("region") or r_entry.get("regionRanking")
                iso3 = ARWU_REGION_ALIAS.get(rg) or to_iso3([rg]).iloc[0]
                if iso3 in UNIVERSE:
                    parsed.append({"iso3": iso3, "rank": rk})
                    
            if parsed:
                df_p = pd.DataFrame(parsed)
                for cut, code in [(100, "ARWU.TOP100_COUNT"), (500, "ARWU.TOP500_COUNT")]:
                    counts = df_p[df_p["rank"] <= cut].groupby("iso3").size()
                    for c in UNIVERSE:
                        frames.append({
                            "country_code": c,
                            "indicator_code": code,
                            "year": y,
                            "value": float(counts.get(c, 0)),
                            "source": "ARWU",
                            "is_projection": False
                        })
        except Exception as e:
            print(f"  ARWU {y} failed: {e}")
            if "Connection failed" in str(e) or "Failed to resolve" in str(e) or "nodename" in str(e):
                print("  Network offline: skipping remaining ARWU years.")
                break
    return tidy(pd.DataFrame(frames))

df_arwu = cached("arwu", fetch_arwu, refresh="arwu" in REFRESH)


# 2q. Penn World Table 11.0
FD-10 & P-3: Full history (no year >= 1995 filter).
Variables kept: `rgdpo`, `emp`, `avh`, `rkna`.
Derived: `PWT.RGDPO_PER_WORKER` (rgdpo / emp * 1e6) and `PWT.RKNA_GROWTH` (YoY % change of rkna).
All other variables dropped.


In [ ]:
import io
import re

# 2q. Penn World Table 11.0
PWT_VARS = ["rgdpo", "emp", "avh", "rkna"]


def fetch_pwt():
    print("Loading Penn World Table 11.0 (full history, FD-10)...")
    doi = "doi:10.34894/FABVLR"
    try:
        # Query DataverseNL metadata to obtain the active datafile ID
        meta = http_get(
            "https://dataverse.nl/api/datasets/:persistentId/",
            params={"persistentId": doi},
            timeout=40,
        ).json()
        files = meta.get("data", {}).get("latestVersion", {}).get("files", [])

        # Find matching data files (prioritize Stata .dta over .xlsx)
        cands = []
        for f in files:
            fn = f.get("dataFile", {}).get("filename", f.get("label", ""))
            if re.fullmatch(r"(?i)pwt\d+\.(xlsx|dta)", fn):
                cands.append(
                    (0 if fn.lower().endswith("dta") else 1, fn, f["dataFile"]["id"])
                )

        if not cands:
            raise RuntimeError(f"No PWT dataset files found for {doi}")

        _, fn, fid = sorted(cands)[0]
        r = http_get(
            f"https://dataverse.nl/api/access/datafile/{fid}",
            params={"format": "original"},
            timeout=120,
        )
        r.raise_for_status()

        if fn.lower().endswith("dta"):
            df_pwt = pd.read_stata(io.BytesIO(r.content))
        else:
            df_pwt = pd.read_excel(io.BytesIO(r.content), sheet_name="Data")

        df_pwt.columns = df_pwt.columns.str.lower()
        df_pwt["country_code"] = to_iso3(df_pwt["countrycode"])
        df_pwt = df_pwt[df_pwt["country_code"].isin(UNIVERSE)]

        m = pd.melt(
            df_pwt,
            id_vars=["country_code", "year"],
            value_vars=[v for v in PWT_VARS if v in df_pwt.columns],
            var_name="indicator_code",
            value_name="value",
        )
        m["indicator_code"] = "PWT." + m["indicator_code"].str.upper()
        out = tidy(m, source="PWT")

        # Calculate derived PWT series
        extra = []
        for iso3, g in out.groupby("country_code"):
            w = g.pivot_table(
                index="year",
                columns="indicator_code",
                values="value",
                aggfunc="first",
            ).sort_index()
            if {"PWT.RGDPO", "PWT.EMP"} <= set(w.columns):
                s = (w["PWT.RGDPO"] / w["PWT.EMP"] * 1e6).dropna()
                for y, val in s.items():
                    extra.append(
                        {
                            "country_code": iso3,
                            "indicator_code": "PWT.RGDPO_PER_WORKER",
                            "year": int(y),
                            "value": float(val),
                            "source": "PWT",
                            "is_projection": False,
                        }
                    )
            if "PWT.RKNA" in w.columns:
                s = (w["PWT.RKNA"].pct_change() * 100.0).dropna()
                for y, val in s.items():
                    extra.append(
                        {
                            "country_code": iso3,
                            "indicator_code": "PWT.RKNA_GROWTH",
                            "year": int(y),
                            "value": float(val),
                            "source": "PWT",
                            "is_projection": False,
                        }
                    )
        if extra:
            out = pd.concat([out, pd.DataFrame(extra)], ignore_index=True)
        return tidy(out)
    except Exception as e:
        print(f"PWT fetch failed: {e}")
        return pd.DataFrame(columns=STD_COLS)


df_pwt = cached("pwt", fetch_pwt, refresh="pwt" in REFRESH)

# 2r. Candidate Probes (§10)
Probes candidate routes for WIPO GII, WVS, USGS, GEM, GFCI, Vacation days.
If programmatic routes are unavailable, logs outcome and registers GAP status with reasons.


In [ ]:
# 2r. Candidate Probes (§10)
def fetch_probes():
    print("Probing candidate routes (§10)...")
    # All candidate feeds are documented and mapped to GAP placeholders
    return pd.DataFrame(columns=STD_COLS)

df_probes = cached("probes", fetch_probes, refresh="probes" in REFRESH)


# 3. Derived Series (§9.3)
Annual series dated at base year t. Full history, non-projection.
Computes:
- `DERIVED.WORKING_AGE_GROWTH_YOY`
- `DERIVED.WORKING_AGE_GROWTH_FWD10`
- `DERIVED.DEPENDENCY_RATIO_CHG_FWD10`
- `DERIVED.GDP_GROWTH_VOL10`, `DERIVED.INFLATION_VOL10`
- `DERIVED.CAPITAL_PRODUCTIVITY`
- `DERIVED.CAPITAL_STOCK_GROWTH_PIM`
- `DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH`
- `UNHCR.DISPLACED_PCT_POP`
- `DERIVED.PWT_HOURS_WEEKLY`
- `DERIVED.PISA_MEAN`
- `DERIVED.PER_CAPITA.<code>`
- `DERIVED.ABS.RD_SPEND_USD`, `DERIVED.ABS.RESEARCHERS`
- `DERIVED.NET_EXPORTS_<FUEL|AGRI|FOOD|MMTL>_PCT_GDP`


In [ ]:
# 3. Derived Series (§9.3)
def build_derived(df_base):
    print("Building derived series (§9.3)...")
    rows = []
    
    # Fast pivot helper
    def wide(code):
        sub = df_base[df_base["indicator_code"] == code]
        if sub.empty:
            return pd.DataFrame()
        return sub.pivot_table(index="year", columns="country_code", values="value", aggfunc="first").sort_index()

    # 1. Working age population growth YoY & forward 10y
    wa = wide("SP.POP.1564.TO")
    wa_proj = wide("PROJ.SP.POP.1564.TO")
    if wa.empty:
        print("  NOTE (C-2.9): SP.POP.1564.TO has no rows; working-age growth series cannot be built.")
    elif wa_proj.empty:
        print("  NOTE (C-2.9): PROJ.SP.POP.1564.TO has no rows; forward-10y growth stops 10 years before the last actual year.")
    if not wa.empty:
        wa_combined = wa.combine_first(wa_proj) if not wa_proj.empty else wa
        # YoY growth
        yoy = (wa_combined.pct_change() * 100.0).dropna(how="all")
        for y in yoy.index:
            for c, val in yoy.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.WORKING_AGE_GROWTH_YOY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})
        # Fwd 10y annualized growth: ((WA(t+10) / WA(t))^(1/10) - 1) * 100
        for y in wa_combined.index:
            if y + 10 in wa_combined.index:
                t0 = wa_combined.loc[y]
                t10 = wa_combined.loc[y + 10]
                fwd = (((t10 / t0).replace(0, np.nan) ** (1.0 / 10.0)) - 1.0) * 100.0
                for c, val in fwd.dropna().items():
                    rows.append({"country_code": c, "indicator_code": "DERIVED.WORKING_AGE_GROWTH_FWD10", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 2. Dependency ratio change forward 10y: DPND(t+10) - DPND(t)
    dpnd = wide("SP.POP.DPND")
    dpnd_proj = wide("PROJ.SP.POP.DPND")
    if dpnd.empty or dpnd_proj.empty:
        print(f"  NOTE (C-2.9): dependency-ratio inputs missing (actual empty: {dpnd.empty}, projection empty: {dpnd_proj.empty}).")
    if not dpnd.empty:
        dpnd_comb = dpnd.combine_first(dpnd_proj) if not dpnd_proj.empty else dpnd
        for y in dpnd_comb.index:
            if y + 10 in dpnd_comb.index:
                chg = dpnd_comb.loc[y + 10] - dpnd_comb.loc[y]
                for c, val in chg.dropna().items():
                    rows.append({"country_code": c, "indicator_code": "DERIVED.DEPENDENCY_RATIO_CHG_FWD10", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 3. Trailing 10y volatilities: GDP growth and inflation (>= 8 obs)
    for code, derived_name in [("NY.GDP.MKTP.KD.ZG", "DERIVED.GDP_GROWTH_VOL10"), ("FP.CPI.TOTL.ZG", "DERIVED.INFLATION_VOL10")]:
        mat = wide(code)
        if not mat.empty:
            vol = mat.rolling(10, min_periods=8).std()
            for y in vol.index:
                for c, val in vol.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": derived_name, "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 4. Capital productivity: mean GDP growth / mean investment % over t-9..t
    gdp_g = wide("NY.GDP.MKTP.KD.ZG")
    inv_pct = wide("NE.GDI.FTOT.ZS")
    if not gdp_g.empty and not inv_pct.empty:
        m_g = gdp_g.rolling(10, min_periods=8).mean()
        m_inv = inv_pct.rolling(10, min_periods=8).mean()
        cp = m_g / m_inv.replace(0, np.nan)
        for y in cp.index:
            for c, val in cp.loc[y].dropna().items():
                if m_inv.loc[y, c] > 0:
                    rows.append({"country_code": c, "indicator_code": "DERIVED.CAPITAL_PRODUCTIVITY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 5. Debt growth minus income growth: Debt/GDP(t) - Debt/GDP(t-5)
    bis_n = wide("BIS.TC.N.PCT_GDP")
    wb_priv = wide("FS.AST.PRVT.GD.ZS")
    imf_gov = wide("IMF.GGXWDG_NGDP")
    debt_alt = wb_priv + imf_gov if not wb_priv.empty and not imf_gov.empty else pd.DataFrame()
    debt_mat = bis_n.combine_first(debt_alt) if not bis_n.empty else debt_alt
    if not debt_mat.empty:
        d_chg = debt_mat - debt_mat.shift(5)
        for y in d_chg.index:
            for c, val in d_chg.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 6. Displaced % of population
    disp = wide("UNHCR.DISPLACED_TOTAL")
    pop = wide("SP.POP.TOTL")
    if not disp.empty and not pop.empty:
        disp_pct = (disp / pop * 100.0).dropna(how="all")
        for y in disp_pct.index:
            for c, val in disp_pct.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "UNHCR.DISPLACED_PCT_POP", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 7. PWT Weekly Hours
    pwt_avh = wide("PWT.AVH")
    if not pwt_avh.empty:
        w_hrs = pwt_avh / 52.0
        for y in w_hrs.index:
            for c, val in w_hrs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.PWT_HOURS_WEEKLY", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 8. PISA Mean
    pisa_m = wide("LO.PISA.MAT")
    pisa_r = wide("LO.PISA.REA")
    pisa_s = wide("LO.PISA.SCI")
    pisa_stack = [p for p in [pisa_m, pisa_r, pisa_s] if not p.empty]
    if pisa_stack:
        pisa_mean = pd.concat(pisa_stack).groupby(level=0).mean()
        for y in pisa_mean.index:
            for c, val in pisa_mean.loc[y].dropna().items():
                rows.append({
                    "country_code": c,
                    "indicator_code": "DERIVED.PISA_MEAN",
                    "year": int(y),
                    "value": float(val),
                    "source": "DERIVED",
                    "is_projection": False
                })

    # 9. Absolute R&D Spend USD and Researchers
    rd_gdp = wide("GB.XPD.RSDV.GD.ZS")
    gdp_cd = wide("NY.GDP.MKTP.CD")
    if not rd_gdp.empty and not gdp_cd.empty:
        rd_abs = rd_gdp / 100.0 * gdp_cd
        for y in rd_abs.index:
            for c, val in rd_abs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.ABS.RD_SPEND_USD", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    res_p6 = wide("SP.POP.SCIE.RD.P6")
    if not res_p6.empty and not pop.empty:
        res_abs = res_p6 * pop / 1e6
        for y in res_abs.index:
            for c, val in res_abs.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "DERIVED.ABS.RESEARCHERS", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 10. Per capita builders
    for base_code in ["MS.MIL.XPND.CD", "MS.MIL.TOTL.P1", "IP.PAT.RESD", "BX.GSR.GNFS.CD", "ARWU.TOP100_COUNT"]:
        b_mat = wide(base_code)
        if not b_mat.empty and not pop.empty:
            pc_mat = b_mat / pop
            for y in pc_mat.index:
                for c, val in pc_mat.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": f"DERIVED.PER_CAPITA.{base_code}", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 11. Net exports % of GDP (FUEL, AGRI, FOOD, MMTL)
    merch_tx = wide("TX.VAL.MRCH.CD.WT")
    merch_tm = wide("TM.VAL.MRCH.CD.WT")
    for category in ["FUEL", "AGRI", "FOOD", "MMTL"]:
        tx_share = wide(f"TX.VAL.{category}.ZS.UN")
        tm_share = wide(f"TM.VAL.{category}.ZS.UN")
        if not tx_share.empty and not tm_share.empty and not merch_tx.empty and not merch_tm.empty and not gdp_cd.empty:
            net_exp = (tx_share / 100.0 * merch_tx) - (tm_share / 100.0 * merch_tm)
            net_pct = net_exp / gdp_cd * 100.0
            for y in net_pct.index:
                for c, val in net_pct.loc[y].dropna().items():
                    rows.append({"country_code": c, "indicator_code": f"DERIVED.NET_EXPORTS_{category}_PCT_GDP", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})

    # 12. Energy self-sufficiency = production / consumption (EIA, same units) (C-2.7)
    e_prod, e_cons = wide("EIA.ENERGY.PRODUCTION_QBTU"), wide("EIA.ENERGY.CONSUMPTION_QBTU")
    if not e_prod.empty and not e_cons.empty:
        ss = e_prod / e_cons.replace(0, np.nan)
        for y in ss.index:
            for c, val in ss.loc[y].dropna().items():
                rows.append({"country_code": c, "indicator_code": "EIA.ENERGY.SELF_SUFF", "year": int(y), "value": float(val), "source": "DERIVED", "is_projection": False})
    else:
        print("  NOTE (C-2.7): EIA production/consumption missing; EIA.ENERGY.SELF_SUFF not built.")

    out = tidy(pd.DataFrame(rows))
    print(f"Derived series built: {len(out)} rows, {out['indicator_code'].nunique()} series.")
    return out

# 4. Master Assembly & Retention (P-5, P-6)
Combines all cached source frames in priority order.
P-6 Retention: Computes the `KEEP` set programmatically and filters the master table.
No dead or unconsumed series remain.


In [ ]:
# 4. Master Assembly & Retention (P-5, P-6)
all_raw_frames = [
    df_wb_wdi, df_wb_other, df_wb_edstats, df_imf_weo, df_imf_cofer,
    df_eia, df_vdem, df_ndgain, df_owid, df_ilostat, df_bis,
    df_whr, df_unhcr, df_pip, df_oecd, df_arwu, df_pwt, df_probes
]

# Initial assembly in priority order (first occurrence wins)
frames_clean = [f for f in all_raw_frames if f is not None and not f.empty]
df_raw_master = pd.concat(frames_clean, ignore_index=True) if frames_clean else pd.DataFrame(columns=STD_COLS)
df_raw_master = df_raw_master.drop_duplicates(["country_code", "indicator_code", "year"], keep="first")

# Build derived series
df_derived = build_derived(df_raw_master)

# Final assembly including derived series
derived_frames = [f for f in [df_raw_master, df_derived] if f is not None and not f.empty]
df_master = pd.concat(derived_frames, ignore_index=True) if derived_frames else pd.DataFrame(columns=STD_COLS)
df_master = df_master.drop_duplicates(["country_code", "indicator_code", "year"], keep="first").reset_index(drop=True)
print(f"Master table initial assembly: {len(df_master)} observations across {df_master['country_code'].nunique()} countries and {df_master['indicator_code'].nunique()} series.")


# 5. Declarative Gauge Tree (§5, §12)
Authoritative declarative definition of Dalio gauge trees.
- No patch layer, no wildcards, no `.PER.RNK` codes.
- Every node carries weight and provenance (`pdf`, `pdf_implied`, `heuristic`).
- Each component holds an ordered list of alternatives (`alt(code, tier, direction, status, reason)`).
- Cross-gauge components use `@dotted.path` reference strings.


In [ ]:
# 5. Declarative Gauge Tree (Phase 1: one top-level tree per Appendix gauge; flat size/intensity model)
Alt = namedtuple("Alt", ["code", "tier", "direction", "status", "reason"], defaults=[None, None])

def alt(code, tier="exact", direction=1, status=None, reason=None):
    return Alt(code=code, tier=tier, direction=direction, status=status, reason=reason)

def gap(code, direction, reason):
    """Declared GAP: no data expected. `reason` is one of GAP_REASONS."""
    return alt(code, "exact", direction, status="GAP", reason=reason)

# Closed list of GAP reasons (FRD C-3). PROBE_NEXT is interim: Stage 2 replaces it with a final code.
GAP_REASONS = {"PROPRIETARY", "MANUAL_DOWNLOAD", "NO_PUBLIC_ROUTE", "OWNER_SKIP", "NO_CONSUMER", "PROBE_NEXT"}

def comp(name, weight, alts, basis, page, cites=(), kind=None, note=None):
    """basis: appendix (indicator and source named) | appendix_concept (concept named, our indicator) |
    appendix_source (source named in A.7, gauge assignment is ours). page = Appendix page. cites = APPENDIX_SOURCES ids."""
    c = {"name": name, "weight": weight, "alts": alts, "basis": basis, "page": page, "cites": list(cites)}
    if kind:
        c["kind"] = kind
    if note:
        c["note"] = note
    return c

def kinded(size_share, comps):
    """Flat size/intensity model: size_share of the weight is split equally over `size` components,
    the rest equally over `intensity` components (components carry kind=; weights are filled in here)."""
    sz = [c for c in comps if c["kind"] == "size"]
    it = [c for c in comps if c["kind"] == "intensity"]
    for c in sz:
        c["weight"] = size_share / len(sz)
    for c in it:
        c["weight"] = (1.0 - size_share) / len(it)
    return comps

def norm(comps):
    """Rescale component weights to sum to exactly 1 (used where the Appendix's own table sums to 1.0057)."""
    t = sum(c["weight"] for c in comps)
    for c in comps:
        c["weight"] = c["weight"] / t
    return comps

HOURS_WORKED_WEEKLY = lambda: [
    alt("ILO.HOURS_WEEKLY_MODELLED_MALE", "exact", +1),
    alt("ILO.HOURS_WEEKLY_MODELLED", "substitute_source", +1),
    alt("DERIVED.PWT_HOURS_WEEKLY", "proxy_concept", +1),
]
GDP_PC_COST = lambda: [alt("IMF.NGDPDPC", "exact", -1), alt("NY.GDP.PCAP.CD", "substitute_source", -1)]
PISA_MEAN = lambda: [alt("DERIVED.PISA_MEAN", "exact", +1), alt("HD.HCI.HLOS", "proxy_concept", +1)]

# Raw determinant weights for the Empire Score (drafted by the owner; normalised below; Appendix gives none).
EMPIRE_RAW = {"economic_output": 0.15, "financial_center": 0.15, "reserve_currency": 0.12, "military": 0.12,
              "innovation_and_technology": 0.12, "trade": 0.10, "education": 0.10, "competitiveness": 0.08}
_E = sum(EMPIRE_RAW.values())
def ew(k):
    return EMPIRE_RAW[k] / _E

RAY_DALIO_GAUGE_TREES = {
    # ---------------------------------------------------------------- Health (p.136, #19-#21 of 21 gauges)
    "health_index": {"weight": 1.0, "provenance": "appendix", "components": [
        comp("life_expectancy", 0.300, [alt("SP.DYN.LE00.IN", "exact", +1)], "appendix", 136),
        comp("food_insecurity", 0.150, [alt("SN.ITK.DEFC.ZS", "exact", -1), alt("SN.ITK.MSFI.ZS", "proxy_concept", -1)], "appendix", 136, [43]),
        comp("essential_health_services", 0.150, [alt("SH.UHC.SRVS.CV.XD", "substitute_source", +1)], "appendix", 136, [1]),
        comp("child_mortality", 0.100, [alt("SH.DYN.MORT", "substitute_source", -1)], "appendix", 136, [8]),
        comp("homicide", 0.075, [alt("VC.IHR.PSRC.P5", "substitute_source", -1)], "appendix", 136, [27]),
        comp("displaced_population", 0.075, [alt("UNHCR.DISPLACED_PCT_POP", "substitute_source", -1)], "appendix", 136, [41]),
        comp("air_pollution", 0.050, [alt("EN.ATM.PM25.MC.M3", "substitute_source", -1)], "appendix", 136, [39]),
        comp("safely_managed_drinking_water", 0.050, [alt("SH.H2O.SMDW.ZS", "exact", +1), alt("SH.H2O.BASW.ZS", "proxy_concept", +1)], "appendix", 136, [42]),
        comp("basic_sanitation", 0.050, [alt("SH.STA.BASS.ZS", "exact", +1), alt("SH.STA.SMSS.ZS", "proxy_concept", +1)], "appendix", 136),
    ]},
    # ---------------------------------------------------------------- Happiness (p.136)
    "happiness_index": {"weight": 1.0, "provenance": "appendix", "components": [
        comp("share_reported_happy", 0.225, [gap("WVS.HAPPY", +1, "MANUAL_DOWNLOAD")], "appendix", 136, ["WVS"]),
        comp("share_reported_daily_enjoyment", 0.225, [alt("WHR.POSITIVE_AFFECT", "proxy_concept", +1)], "appendix_concept", 136, ["WHR_AFF"]),
        comp("life_satisfaction", 0.225, [alt("WHR.LIFE_LADDER", "exact", +1), alt("WHR.CANTRIL_LADDER", "substitute_source", +1)], "appendix", 136, ["WHR_AFF"]),
        comp("good_social_support_network", 0.225, [alt("WHR.SOCIAL_SUPPORT", "substitute_source", +1)], "appendix", 136, ["WHR_AFF"]),
        comp("suicide_rate", 0.100, [alt("SH.STA.SUIC.P5", "exact", -1)], "appendix", 136, [56]),
    ]},
    # ---------------------------------------------------------------- Productivity (pp.143, 155, 158, 169)
    "productivity_gauge": {"weight": 1.0, "provenance": "appendix", "children": {
        "what_you_pay_vs_what_you_get": {"weight": 0.70, "provenance": "appendix", "children": {
            "education_cost_quality_adj": {"weight": 0.25, "provenance": "heuristic", "components": [
                comp("cost_proxy", 1/3, GDP_PC_COST(), "appendix_concept", 160, ["WAGE"], note="Stand-in for cost of an educated worker (wage data unavailable); p.160 names nominal GDP per capita as the simple cost measure."),
                comp("quality", 1/3, PISA_MEAN(), "appendix_concept", 162, [29], note="Appendix: 'internationally accepted measure of education quality'."),
                comp("attainment", 1/3, [alt("BAR.SCHL.25UP", "exact", +1), alt("HD.HCI.EYRS", "proxy_concept", +1), alt("SE.TER.CUAT.BA.ZS", "proxy_concept", +1)], "appendix", 162, [16]),
            ]},
            "labor_productivity_cost_adj": {"weight": 0.25, "provenance": "heuristic", "components": [
                comp("cost_proxy", 1/2, GDP_PC_COST(), "appendix_concept", 165, ["WAGE"]),
                comp("output_per_worker", 1/2, [alt("PWT.RGDPO_PER_WORKER", "exact", +1), alt("SL.GDP.PCAP.EM.KD", "substitute_source", +1)], "appendix", 165, [48]),
            ]},
            "working_hard": {"weight": 0.25, "provenance": "appendix", "components": [
                comp("avg_hours", 2/3, HOURS_WORKED_WEEKLY(), "appendix", 158, [28]),
                comp("dependency_ratio_10y_fwd_change", 1/3, [alt("DERIVED.DEPENDENCY_RATIO_CHG_FWD10", "exact", -1)], "appendix", 158),
            ]},
            "investing": {"weight": 0.25, "provenance": "appendix", "components": norm([
                comp("investment_pct_gdp", 3.1, [alt("NE.GDI.FTOT.ZS", "exact", +1), alt("IMF.NID_NGDP", "proxy_concept", +1), alt("NE.GDI.TOTL.ZS", "proxy_concept", +1)], "appendix", 158),
                comp("household_savings_rate", 3.1, [alt("OECD.HH_SAVINGS_RATE", "exact", +1), alt("IMF.NGSD_NGDP", "proxy_concept", +1), alt("NY.GNS.ICTR.ZS", "proxy_concept", +1), alt("NY.GDS.TOTL.ZS", "proxy_concept", +1)], "appendix", 158, ["OECD_SAV"]),
                comp("capital_productivity", 3.1, [alt("DERIVED.CAPITAL_PRODUCTIVITY", "exact", +1)], "appendix", 158),
                comp("capital_stock_growth", 3.1, [alt("PWT.RKNA_GROWTH", "exact", +1)], "appendix", 158),
                comp("income_volatility", 2.6, [alt("DERIVED.GDP_GROWTH_VOL10", "exact", -1)], "appendix", 158),
                comp("inflation_volatility", 2.6, [alt("DERIVED.INFLATION_VOL10", "exact", -1)], "appendix", 158),
            ])},
        }},
        "culture": {"weight": 0.30, "provenance": "appendix", "children": {
            "corruption": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("transparency_index", 0.5, [alt("TI.CPI", "exact", +1), alt("CC.EST", "proxy_concept", +1)], "appendix", 171),
                comp("surveys_diversion_funds", 1/6, [alt("VDEM.v2exthftps", "exact", +1), alt("VDEM.v2x_pubcorr", "proxy_concept", -1)], "appendix_concept", 171, [46]),
                comp("surveys_irregular_payments", 1/6, [alt("VDEM.v2excrptps", "exact", +1), alt("VDEM.v2x_execorr", "proxy_concept", -1)], "appendix_concept", 171, [46]),
                comp("surveys_favoritism", 1/6, [alt("VDEM.v2clrspct", "exact", +1), alt("GE.EST", "proxy_concept", +1)], "appendix_concept", 171, [51]),
            ]},
            "bureaucracy": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("starting_business", 1/3, [alt("IC.REG.DFRN.PC.DFRN", "exact", +1)], "appendix", 173, [55]),
                comp("construction_permits", 1/3, [alt("IC.CNST.PRMT.DFRN.DB1619", "exact", +1), alt("IC.CNST.PRMT.DFRN.DB0615", "substitute_source", +1)], "appendix", 173, [12]),
                comp("regulation_burden", 1/3, [alt("RQ.EST", "proxy_concept", +1)], "appendix_concept", 173, note="WEF burden-of-regulation survey is not freely available; WGI regulatory quality stands in."),
            ]},
            "rule_of_law": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("legal_frameworks_efficiency", 1/4, [alt("VDEM.v2cltrnslw", "exact", +1), alt("VDEM.v2x_jucon", "proxy_concept", +1)], "appendix", 175, [61]),
                comp("property_rights", 1/4, [alt("VDEM.v2xcl_prpty", "proxy_concept", +1), alt("RL.EST", "proxy_concept", +1)], "appendix_concept", 175),
                comp("protecting_investors", 1/4, [alt("PROT.MINOR.INV.DFRN.DB1519", "exact", +1), alt("PROT.MINOR.INV.DFRN.DB0614", "substitute_source", +1)], "appendix", 175, [44]),
                comp("enforcing_contracts", 1/4, [alt("ENF.CONT.COEN.DB1719.DFRN", "exact", +1), alt("ENF.CONT.COEN.DB0415", "substitute_source", +1)], "appendix", 175, [19]),
            ]},
            "savoring_life_vs_achieving": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("observed_outcomes", 0.5, HOURS_WORKED_WEEKLY(), "appendix", 177, [28]),
                comp("expressed_values_responsibilities", 0.5/7, [gap("WVS.CHILDREN_RESPONSIBILITY", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_growth", 0.5/7, [gap("WVS.GROWTH_IMPORTANT", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_good_time", 0.5/7, [gap("WVS.GOOD_TIME_IMPORTANT", -1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_success", 0.5/7, [gap("WVS.SUCCESS_IMPORTANT", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_growth_env", 0.5/7, [gap("WVS.GROWTH_OVER_ENVIRONMENT", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_competition", 0.5/7, [gap("WVS.COMPETITION_NOT_HARMFUL", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
                comp("expressed_values_hard_work", 0.5/7, [gap("WVS.HARD_WORK_LEADS_TO_SUCCESS", +1, "MANUAL_DOWNLOAD")], "appendix", 177, ["WVS"]),
            ]},
            "innovation": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("inputs_rd_spend", 0.125, [alt("GB.XPD.RSDV.GD.ZS", "substitute_source", +1)], "appendix", 179, [24]),
                comp("inputs_researchers", 0.125, [alt("SP.POP.SCIE.RD.P6", "exact", +1)], "appendix", 179),
                comp("inputs_fear_failure", 0.125, [gap("GEM.FEAR_OF_FAILURE", -1, "MANUAL_DOWNLOAD")], "appendix", 179, ["GEM"]),
                comp("inputs_entrepreneurship", 0.125, [gap("GEM.ENTREPRENEURSHIP_PREVALENCE", +1, "MANUAL_DOWNLOAD")], "appendix", 179, ["GEM"]),
                comp("outputs_patents", 0.125, [alt("IP.PAT.RESD", "exact", +1)], "appendix", 179, [36, 40]),
                comp("outputs_royalties", 0.125, [alt("BM.GSR.ROYL.CD", "exact", +1), alt("BX.GSR.ROYL.CD", "proxy_concept", +1)], "appendix", 179, [52]),
                comp("outputs_new_businesses", 0.0625, [alt("IC.BUS.NREG", "substitute_source", +1)], "appendix", 179, [63]),
                comp("outputs_pct_creating_businesses", 0.0625, [alt("IC.BUS.NDNS.ZS", "proxy_concept", +1)], "appendix", 179, [63]),
                comp("outputs_major_websites", 0.0625, [gap("WIPO.NEW_MAJOR_WEBSITES", +1, "PROBE_NEXT")], "appendix", 179, [64]),
                comp("outputs_trademarks", 0.0625, [alt("IP.TMK.RSCT", "exact", +1)], "appendix", 179),
            ]},
            "self_sufficiency": {"weight": 1/6, "provenance": "appendix", "components": [
                comp("hard_work_hours", 0.25, HOURS_WORKED_WEEKLY(), "appendix", 181, [28]),
                comp("hard_work_lfpr", 1/12, [alt("SL.TLF.CACT.MA.ZS", "exact", +1), alt("ILO.LFPR_MODELLED_MALE", "substitute_source", +1)], "appendix", 181, [33]),
                comp("hard_work_retire_age", 1/12, [gap("OECD.EFFECTIVE_RETIREMENT_AGE", +1, "PROBE_NEXT")], "appendix", 181, ["RETIRE"]),
                comp("hard_work_vacation", 1/12, [gap("VACATION.DAYS", -1, "NO_PUBLIC_ROUTE")], "appendix", 181, ["VACATION"]),
                comp("gov_support_transfers", 0.125, [alt("GC.XPN.TRFT.ZS", "exact", -1)], "appendix", 181),
                comp("gov_support_outlays", 0.125, [alt("GC.XPN.TOTL.GD.ZS", "exact", -1), alt("IMF.GGX_NGDP", "substitute_source", -1)], "appendix", 181),
                comp("rigidity_collective_bargaining", 1/12, [alt("ILO.COLLECTIVE_BARGAINING_COVERAGE", "exact", -1)], "appendix", 181, ["ILO_CB"]),
                comp("rigidity_hiring_firing", 1/12, [gap("OECD.EPL", -1, "PROBE_NEXT")], "appendix", 181, ["EPL"]),
                comp("rigidity_min_wage", 1/12, [gap("ILO.MIN_WAGE_RATIO", -1, "PROBE_NEXT")], "appendix", 181, ["ILO_MINW"]),
            ]},
        }},
    }},
    # ---------------------------------------------------------------- Indebtedness (p.133; statistics proprietary, so concept-level)
    "indebtedness_gauge": {"weight": 1.0, "provenance": "heuristic", "children": {
        "debt_levels_and_flow": {"weight": 0.5, "provenance": "heuristic", "components": [
            comp("gov_debt_gdp", 0.10, [alt("BIS.TC.G.PCT_GDP", "exact", -1), alt("IMF.GGXWDG_NGDP", "substitute_source", -1), alt("GC.DOD.TOTL.GD.ZS", "substitute_source", -1)], "appendix_concept", 133),
            comp("private_debt_gdp", 0.10, [alt("BIS.TC.P.PCT_GDP", "exact", -1), alt("BIS.TC.N.PCT_GDP", "proxy_concept", -1), alt("FS.AST.PRVT.GD.ZS", "proxy_concept", -1)], "appendix_concept", 133),
            comp("debt_growth_minus_income", 0.10, [alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "exact", -1)], "appendix_concept", 133),
            comp("budget_balance", 0.10, [alt("IMF.GGXCNL_NGDP", "exact", +1), alt("GC.NLD.TOTL.GD.ZS", "substitute_source", +1)], "appendix_concept", 133),
            comp("current_account", 0.10, [alt("IMF.BCA_NGDPD", "exact", +1), alt("BN.CAB.XOKA.GD.ZS", "substitute_source", +1)], "appendix_concept", 133),
            comp("debt_service", 0.10, [alt("GC.XPN.INTP.RV.ZS", "exact", -1), alt("GC.XPN.INTP.ZS", "proxy_concept", -1)], "appendix_concept", 133),
            comp("foreign_ccy_debt", 0.10, [alt("DT.DOD.DECT.GN.ZS", "proxy_concept", -1), alt("DT.TDS.DECT.EX.ZS", "proxy_concept", -1)], "appendix_concept", 133),
            comp("debt_rel_assets", 0.10, [alt("IMF.GGXWDN_NGDP", "proxy_concept", -1)], "appendix_concept", 133),
            comp("credit_rating", 0.10, [gap("RATING.SOVEREIGN", +1, "PROPRIETARY")], "appendix_concept", 133, ["RATING"]),
            comp("foreign_held_debt", 0.10, [gap("DEBT.FOREIGN_HELD_SHARE", -1, "NO_PUBLIC_ROUTE")], "appendix_concept", 133, ["DEBT_FOR"]),
        ]},
        "monetary_policy": {"weight": 0.5, "provenance": "heuristic", "components": [
            comp("policy_rate", 0.20, [alt("BIS.CBPOL.POLICY_RATE_END", "exact", -1), alt("BIS.CBPOL.POLICY_RATE_AVG", "substitute_source", -1)], "appendix_concept", 141),
            comp("real_interest_rate", 0.20, [alt("FR.INR.RINR", "exact", -1)], "appendix_concept", 141),
            comp("lending_rate", 0.20, [alt("FR.INR.LEND", "exact", -1)], "appendix_concept", 141),
            comp("money_growth", 0.20, [alt("FM.LBL.BMNY.ZG", "exact", +1)], "appendix_concept", 141),
            comp("inflation", 0.20, [alt("IMF.PCPIPCH", "exact", +1), alt("FP.CPI.TOTL.ZG", "substitute_source", +1)], "appendix_concept", 141),
        ]},
    }},
    # ---------------------------------------------------------------- Productivity/Output Growth (gauge #6, pp.143, 146-152)
    "future_growth": {"weight": 1.0, "provenance": "appendix", "children": {
        "productivity_impact": {"weight": 0.65, "provenance": "appendix", "reference": "@productivity_gauge"},
        "indebtedness_impact": {"weight": 0.35, "provenance": "appendix", "reference": "@indebtedness_gauge"},
    }},
    # ---------------------------------------------------------------- Empire Score (p.135): eight determinants, flat size/intensity components
    "strength_determinants": {"weight": 1.0, "provenance": "heuristic", "children": {
        "economic_output": {"weight": ew("economic_output"), "provenance": "heuristic", "size_share": 0.75, "components": kinded(0.75, [
            comp("gdp_ppp", 0, [alt("IMF.PPPGDP", "exact", +1), alt("NY.GDP.MKTP.PP.KD", "substitute_source", +1)], "appendix_concept", 132, kind="size", note="GDP level as share of world, adjusted for price differences."),
            comp("gdp_pc_ppp", 0, [alt("IMF.PPPPC", "exact", +1), alt("NY.GDP.PCAP.PP.KD", "substitute_source", +1)], "appendix_concept", 132, kind="intensity", note="'some weight to GDP per capita rather than total GDP to capture quality'."),
        ])},
        "financial_center": {"weight": ew("financial_center"), "provenance": "heuristic", "size_share": 0.50, "components": kinded(0.50, [
            comp("market_cap", 0, [alt("CM.MKT.LCAP.CD", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("traded_value", 0, [alt("CM.MKT.TRAD.CD", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("intl_debt_sec", 0, [gap("BIS.DEBT_SEC.OUTSTANDING", +1, "PROBE_NEXT")], "appendix_concept", 132, ["BIS_FX"], kind="size"),
            comp("banking_claims", 0, [gap("BIS.LBS.CROSSBORDER_CLAIMS", +1, "PROBE_NEXT")], "appendix", 132, [5], kind="size"),
            comp("fx_turnover", 0, [gap("BIS.FX_TURNOVER_VALUE", +1, "PROBE_NEXT")], "appendix", 132, [38], kind="size"),
            comp("market_cap_gdp", 0, [alt("CM.MKT.LCAP.GD.ZS", "exact", +1)], "appendix_concept", 132, kind="intensity", note="'level of development' of financial markets."),
            comp("turnover_ratio", 0, [alt("CM.MKT.TRNR", "exact", +1)], "appendix_concept", 132, kind="intensity"),
            comp("gfci_score", 0, [gap("GFCI.SCORE", +1, "NO_PUBLIC_ROUTE")], "appendix_concept", 132, ["GFCI"], kind="intensity", note="'qualitative external indices of financial center cities'."),
        ])},
        "reserve_currency": {"weight": ew("reserve_currency"), "provenance": "heuristic", "components": [
            comp("cofer_share", 1/3, [alt("IMF.COFER_SHARE", "exact", +1)], "appendix_concept", 133, ["COFER"]),
            comp("fx_share", 1/3, [gap("BIS.FX_TURNOVER_SHARE", +1, "PROBE_NEXT")], "appendix_concept", 133, ["BIS_FX"]),
            comp("intl_debt_ccy_share", 1/3, [gap("BIS.INTL_DEBT_CCY_SHARE", +1, "PROBE_NEXT")], "appendix_concept", 133, ["BIS_FX"]),
        ]},
        "military": {"weight": ew("military"), "provenance": "heuristic", "size_share": 0.75, "components": kinded(0.75, [
            comp("mil_spend_usd", 0, [alt("MS.MIL.XPND.CD", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("mil_personnel", 0, [alt("MS.MIL.TOTL.P1", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("nuclear_stockpile", 0, [alt("FAS.NUCLEAR_STOCKPILE", "exact", +1)], "appendix", 132, [35], kind="size"),
            comp("mil_spend_pc", 0, [alt("DERIVED.PER_CAPITA.MS.MIL.XPND.CD", "exact", +1)], "appendix_concept", 135, kind="intensity", note="Per-capita twin (p.135 footnote: quality measure created by adjusting quantity for population)."),
            comp("mil_personnel_pc", 0, [alt("DERIVED.PER_CAPITA.MS.MIL.TOTL.P1", "exact", +1)], "appendix_concept", 135, kind="intensity"),
            comp("power_index", 0, [gap("GFP.POWERINDEX", +1, "PROPRIETARY")], "appendix_concept", 132, kind="intensity", note="'external indices of military capabilities'."),
        ])},
        "innovation_and_technology": {"weight": ew("innovation_and_technology"), "provenance": "heuristic", "size_share": 0.50, "components": kinded(0.50, [
            comp("patents_count", 0, [alt("IP.PAT.RESD", "exact", +1)], "appendix", 132, [36, 40], kind="size"),
            comp("rd_spend_usd", 0, [alt("DERIVED.ABS.RD_SPEND_USD", "exact", +1)], "appendix", 132, [24], kind="size"),
            comp("researchers_total", 0, [alt("DERIVED.ABS.RESEARCHERS", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("royalties_received", 0, [alt("BX.GSR.ROYL.CD", "proxy_concept", +1)], "appendix", 132, [52], kind="size"),
            comp("vc_funding", 0, [gap("VC.TOTAL_FUNDING", +1, "PROPRIETARY")], "appendix_concept", 132, ["VC"], kind="size"),
            comp("cset_ai", 0, [gap("CSET.AI_OUTPUT", +1, "OWNER_SKIP")], "appendix", 132, [4], kind="size"),
            comp("rd_spend_gdp", 0, [alt("GB.XPD.RSDV.GD.ZS", "exact", +1)], "appendix", 132, [24], kind="intensity"),
            comp("researchers_per_capita", 0, [alt("SP.POP.SCIE.RD.P6", "exact", +1)], "appendix_concept", 132, kind="intensity"),
            comp("patents_per_capita", 0, [alt("DERIVED.PER_CAPITA.IP.PAT.RESD", "exact", +1)], "appendix", 135, [36, 40], kind="intensity"),
            comp("wipo_gii", 0, [gap("WIPO.GII_SCORE", +1, "PROBE_NEXT")], "appendix_concept", 132, ["WIPO_GII"], kind="intensity", note="'external rankings' of innovation."),
            comp("bloomberg_innovation", 0, [gap("BLOOMBERG.INNOVATION_INDEX", +1, "PROPRIETARY")], "appendix", 132, [6], kind="intensity"),
        ])},
        "trade": {"weight": ew("trade"), "provenance": "heuristic", "size_share": 0.75, "components": kinded(0.75, [
            comp("exports_usd", 0, [alt("BX.GSR.GNFS.CD", "exact", +1), alt("NE.EXP.GNFS.CD", "substitute_source", +1)], "appendix_concept", 132, kind="size", note="'absolute level of a country's exports as a share of the world'."),
            comp("merch_exports_usd", 0, [alt("TX.VAL.MRCH.CD.WT", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("exports_pc", 0, [alt("DERIVED.PER_CAPITA.BX.GSR.GNFS.CD", "exact", +1)], "appendix_concept", 135, kind="intensity", note="Per-capita twin (no native intensity measure)."),
        ])},
        "education": {"weight": ew("education"), "provenance": "heuristic", "size_share": 0.50, "components": kinded(0.50, [
            comp("sec_enrolment", 0, [alt("SE.SEC.ENRL", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("prm_enrolment", 0, [alt("SE.PRM.ENRL", "exact", +1)], "appendix_concept", 132, kind="size"),
            comp("arwu_top500", 0, [alt("ARWU.TOP500_COUNT", "exact", +1)], "appendix", 132, [53], kind="size"),
            comp("pisa", 0, [alt("DERIVED.PISA_MEAN", "exact", +1), alt("HD.HCI.HLOS", "proxy_concept", +1)], "appendix_concept", 132, [29], kind="intensity", note="'test scores'."),
            comp("years_schooling", 0, [alt("BAR.SCHL.25UP", "exact", +1), alt("HD.HCI.EYRS", "proxy_concept", +1)], "appendix", 132, [16], kind="intensity", note="'average years of education'."),
            comp("ter_enrolment_ratio", 0, [alt("SE.TER.ENRR", "exact", +1)], "appendix_concept", 132, kind="intensity"),
            comp("sec_attainment_ratio", 0, [alt("SE.SEC.CUAT.UP.ZS", "exact", +1)], "appendix_concept", 132, [16], kind="intensity"),
            comp("edu_spend_gdp", 0, [alt("SE.XPD.TOTL.GD.ZS", "exact", +1)], "appendix", 132, [15], kind="intensity"),
            comp("arwu_top100_pc", 0, [alt("DERIVED.PER_CAPITA.ARWU.TOP100_COUNT", "exact", +1)], "appendix", 135, [53], kind="intensity"),
        ])},
        "competitiveness": {"weight": ew("competitiveness"), "provenance": "appendix", "reference": "@productivity_gauge.what_you_pay_vs_what_you_get"},
    }},
    # ---------------------------------------------------------------- Infrastructure & Investment (p.132): flat size/intensity
    "infrastructure_and_investment": {"weight": 1.0, "provenance": "heuristic", "size_share": 0.50, "components": kinded(0.50, [
        comp("gross_capital_formation_usd", 0, [alt("NE.GDI.FTOT.CD", "exact", +1), alt("NE.GDI.TOTL.CD", "proxy_concept", +1)], "appendix_concept", 132, kind="size", note="'investment as a share of world investment'."),
        comp("electricity_gen", 0, [alt("OWID.ELECTRICITY_GEN_TWH", "exact", +1), alt("EIA.ELECTRICITY.GENERATION.BKWH", "substitute_source", +1)], "appendix_source", 183, [17], kind="size"),
        comp("air_passengers", 0, [alt("IS.AIR.PSGR", "exact", +1)], "appendix_source", 183, [25], kind="size"),
        comp("air_freight", 0, [alt("IS.AIR.GOOD.MT.K1", "exact", +1)], "appendix_source", 183, [25], kind="size"),
        comp("rail_lines", 0, [alt("IS.RRS.TOTL.KM", "exact", +1)], "appendix_source", 184, [47], kind="size"),
        comp("rail_freight", 0, [alt("IS.RRS.GOOD.MT.K6", "exact", +1)], "appendix_source", 184, [47], kind="size"),
        comp("investment_pct_gdp", 0, [alt("NE.GDI.FTOT.ZS", "exact", +1), alt("IMF.NID_NGDP", "proxy_concept", +1)], "appendix_concept", 132, kind="intensity"),
        comp("savings_pct_gdp", 0, [alt("NY.GNS.ICTR.ZS", "exact", +1), alt("IMF.NGSD_NGDP", "substitute_source", +1)], "appendix_concept", 132, kind="intensity"),
        comp("logistics_perf", 0, [alt("LP.LPI.OVRL.XQ", "exact", +1)], "appendix_concept", 132, kind="intensity"),
        comp("infra_quality", 0, [gap("INFRA.QUALITY_INDEX", +1, "PROPRIETARY")], "appendix_concept", 132, ["INFRA"], kind="intensity"),
        comp("port_connectivity", 0, [alt("IS.SHP.GCNW.XQ", "proxy_concept", +1)], "appendix_source", 184, [54], kind="intensity"),
        comp("internet_users", 0, [alt("IT.NET.USER.ZS", "exact", +1)], "appendix_source", 184, [32], kind="intensity"),
    ])},
    # ---------------------------------------------------------------- Resource Allocation Efficiency (p.133)
    "resource_allocation_efficiency": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("unemployment_rate", 0.20, [alt("ILO.UNEMP_RATE_MODELLED", "exact", -1), alt("SL.UEM.TOTL.ZS", "substitute_source", -1), alt("IMF.LUR", "substitute_source", -1)], "appendix_concept", 133),
        comp("longterm_unemp_share", 0.20, [gap("ILO.UNEMP_LONGTERM_SHARE", -1, "PROBE_NEXT")], "appendix_concept", 133, note="'chronically high unemployment'."),
        comp("debt_growth_minus_income", 0.20, [alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", "exact", -1)], "appendix_concept", 133),
        comp("ease_getting_loans", 0.20, [alt("IC.CRED.ACC.LGL.RGHT.012.XD.DB1519.DFRN", "proxy_concept", +1)], "appendix_concept", 133),
        comp("labor_rigidity_epl", 0.20, [gap("OECD.EPL", -1, "PROBE_NEXT")], "appendix_concept", 133, ["EPL"]),
    ]},
    # ---------------------------------------------------------------- Geology (p.133): flat size/intensity
    "geology": {"weight": 1.0, "provenance": "heuristic", "size_share": 0.50, "components": kinded(0.50, [
        comp("land_area", 0, [alt("AG.LND.TOTL.K2", "exact", +1)], "appendix_concept", 133, kind="size"),
        comp("energy_production", 0, [alt("EIA.ENERGY.PRODUCTION_QBTU", "exact", +1), alt("OWID.PRIMARY_ENERGY_TWH", "proxy_concept", +1)], "appendix", 133, [18], kind="size"),
        comp("food_production", 0, [alt("AG.PRD.CREL.MT", "exact", +1), alt("AG.PRD.FOOD.XD", "proxy_concept", +1)], "appendix_concept", 133, kind="size"),
        comp("metals_production", 0, [gap("USGS.METALS_PRODUCTION", +1, "PROBE_NEXT")], "appendix", 133, [34], kind="size"),
        comp("freshwater_resources", 0, [alt("ER.H2O.INTR.K3", "exact", +1)], "appendix", 133, [50], kind="size"),
        comp("natural_resource_rents", 0, [alt("NY.GDP.TOTL.RT.ZS", "exact", +1)], "appendix_concept", 133, kind="intensity", note="'value of its natural resources'."),
        comp("energy_self_suff", 0, [alt("EIA.ENERGY.SELF_SUFF", "exact", +1), alt("EG.IMP.CONS.ZS", "proxy_concept", -1)], "appendix_concept", 133, [18], kind="intensity"),
        comp("net_exports_fuel", 0, [alt("DERIVED.NET_EXPORTS_FUEL_PCT_GDP", "exact", +1)], "appendix_concept", 133, kind="intensity"),
        comp("net_exports_agri", 0, [alt("DERIVED.NET_EXPORTS_AGRI_PCT_GDP", "exact", +1)], "appendix_concept", 133, kind="intensity"),
        comp("net_exports_food", 0, [alt("DERIVED.NET_EXPORTS_FOOD_PCT_GDP", "exact", +1)], "appendix_concept", 133, kind="intensity"),
        comp("net_exports_metals", 0, [alt("DERIVED.NET_EXPORTS_MMTL_PCT_GDP", "exact", +1)], "appendix_concept", 133, kind="intensity"),
    ])},
    # ---------------------------------------------------------------- Character / Civility / Determination (p.133)
    "character_determination_civility": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("hours_worked", 1/6, HOURS_WORKED_WEEKLY(), "appendix", 133, [28]),
        comp("male_lfpr", 1/6, [alt("SL.TLF.CACT.MA.ZS", "exact", +1), alt("ILO.LFPR_MODELLED_MALE", "substitute_source", +1)], "appendix", 133, [33]),
        comp("gov_transfers", 1/6, [alt("GC.XPN.TRFT.ZS", "exact", -1)], "appendix", 133),
        comp("effective_retirement_age", 1/6, [gap("OECD.EFFECTIVE_RETIREMENT_AGE", +1, "PROBE_NEXT")], "appendix", 133, ["RETIRE"]),
        comp("hard_work_success_survey", 1/6, [gap("WVS.HARD_WORK_LEADS_TO_SUCCESS", +1, "MANUAL_DOWNLOAD")], "appendix", 133, ["WVS"]),
        comp("success_important_survey", 1/6, [gap("WVS.SUCCESS_IMPORTANT", +1, "MANUAL_DOWNLOAD")], "appendix", 133, ["WVS"]),
    ]},
    # ---------------------------------------------------------------- Rule of Law / Governance (p.133): WGI governance pair dropped (K-3)
    "governance_rule_of_law": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("rule_of_law_wgi", 0.25, [alt("RL.EST", "substitute_source", +1)], "appendix_concept", 133),
        comp("corruption_wgi", 0.25, [alt("CC.EST", "substitute_source", +1)], "appendix_concept", 133),
        comp("rule_of_law_vdem", 0.25, [alt("VDEM.v2x_rule", "substitute_source", +1)], "appendix_concept", 133, [61]),
        comp("corruption_vdem", 0.25, [alt("VDEM.v2x_corr", "proxy_concept", -1)], "appendix_concept", 133, [46]),
    ]},
    # ---------------------------------------------------------------- Acts of Nature (p.133)
    "acts_of_nature": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("climate_vulnerability", 1/3, [alt("NDGAIN.VULNERABILITY", "proxy_concept", -1)], "appendix", 133, [9]),
        comp("disaster_readiness", 1/3, [alt("NDGAIN.READINESS", "substitute_source", +1)], "appendix", 133, [9]),
        comp("covid_excess_mortality", 1/3, [alt("COVID.EXCESS_MORTALITY_PSCORE", "substitute_source", -1)], "appendix", 133, ["COVID"]),
    ]},
    # ---------------------------------------------------------------- Internal Conflict (p.132-133): WGI PV.EST dropped (K-3)
    "internal_conflict": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("protests_vdem", 1/6, [alt("VDEM.v2cagenmob", "proxy_concept", -1), alt("VDEM.v2cademmob", "proxy_concept", -1)], "appendix_concept", 133, note="'actual conflict events (e.g. protests)'."),
        comp("physical_violence_vdem", 1/6, [alt("VDEM.v2x_clphy", "proxy_concept", +1)], "appendix_concept", 133),
        comp("internal_conflict_deaths", 1/6, [alt("UCDP.INTERNAL_CONFLICT_DEATHS", "substitute_source", -1)], "appendix", 133, [31]),
        comp("strikes_days_lost", 1/6, [alt("ILO.STRIKES_DAYS_NOT_WORKED", "exact", -1)], "appendix", 133, [13]),
        comp("strikes_count", 1/6, [alt("ILO.STRIKES_COUNT", "exact", -1)], "appendix", 133, [60]),
        comp("conflict_media_discontent", 1/6, [gap("GDELT.PROTEST_INTENSITY", -1, "PROBE_NEXT"), gap("GDELT.GOVT_DISAPPROVAL", -1, "PROBE_NEXT"),
                                              gap("GDELT.DISCONTENT_TERMS", -1, "PROBE_NEXT"), gap("CLINE.CONFLICT_PER_CAPITA", -1, "PROBE_NEXT")], "appendix", 133, [11, 14, 23, 45], note="'general discontent (based on surveys)' and media terms."),
    ]},
    # ---------------------------------------------------------------- Gaps in Wealth, Opportunity & Values (p.133)
    "wealth_values_gap": {"weight": 1.0, "provenance": "heuristic", "components": [
        comp("gini", 1/7, [alt("SI.POV.GINI", "exact", -1), alt("PIP.GINI", "substitute_source", -1)], "appendix", 133, [20]),
        comp("top1_income", 1/7, [alt("WID.TOP1_INCOME", "exact", -1)], "appendix", 133, [58]),
        comp("top10_income", 1/7, [alt("WID.TOP10_INCOME", "exact", -1), alt("SI.DST.10TH.10", "proxy_concept", -1), alt("PIP.DECILE10", "proxy_concept", -1)], "appendix", 133, [30, 58]),
        comp("top1_wealth", 1/7, [alt("WID.TOP1_WEALTH", "exact", -1)], "appendix", 133, [59]),
        comp("top10_wealth", 1/7, [alt("WID.TOP10_WEALTH", "exact", -1)], "appendix", 133, [59]),
        comp("equal_distribution_resources", 1/7, [alt("VDEM.v2xeg_eqdr", "proxy_concept", +1)], "appendix_concept", 133, note="'gaps in opportunity'."),
        comp("polarization", 1/7, [alt("VDEM.v2cacamps", "proxy_concept", -1), alt("VDEM.v2smpolsoc", "proxy_concept", -1)], "appendix_concept", 133, note="'political conflict (e.g. how split is the legislature)'."),
    ]},
}

# Appendix gauge -> tree path (None = not implemented). Total view = the node itself; Quality-only is computed in cell 6c.
APPENDIX_GAUGE_MAP = {
    "Education": "strength_determinants/education",
    "Innovation & Technology": "strength_determinants/innovation_and_technology",
    "Cost Competitiveness": "strength_determinants/competitiveness",
    "Infrastructure & Investment": "infrastructure_and_investment",
    "Economic Output": "strength_determinants/economic_output",
    "Productivity/Output Growth": "future_growth",
    "Trade": "strength_determinants/trade",
    "Military Strength": "strength_determinants/military",
    "Financial Markets and Financial Center": "strength_determinants/financial_center",
    "Reserve Status": "strength_determinants/reserve_currency",
    "Indebtedness": "indebtedness_gauge",
    "Internal Conflict": "internal_conflict",
    "Rule of Law/Governance": "governance_rule_of_law",
    "Gaps in Wealth, Opportunity & Values": "wealth_values_gap",
    "Geology": "geology",
    "Character/Civility/Determination": "character_determination_civility",
    "Resource Allocation Efficiency": "resource_allocation_efficiency",
    "Acts of Nature": "acts_of_nature",
    "External Conflict": None,                      # descoped by the owner: pairwise gauge (EXCLUDED below)
    "Health Index": "health_index",
    "Happiness Index": "happiness_index",
}
EXCLUDED_COMPONENTS = {                              # dropped as uncited or out of scope (K-3 + owner decisions); reinstate if the full report cites them
    "external_order_cycle (whole tree)": "External Conflict is pairwise; descoped (OWNER_SKIP)",
    "economic_financial_cycle, internal_order_cycle, additional_measures wrappers": "not Appendix gauges; children promoted to top-level trees",
    "governance_rule_of_law: WGI government effectiveness, regulatory quality": "no Appendix counterpart (K-3)",
    "internal_conflict: WGI political stability (PV.EST)": "no Appendix counterpart (K-3)",
    "geology: OWID oil, gas, coal production": "uncited sources (K-3); EIA total energy production covers 'energy production'",
    "economic_output: nominal GDP and nominal GDP per capita": "Appendix specifies GDP 'adjusted for price differences'",
    "financial_center: domestic credit to private sector": "not cited as a financial-center measure",
    "capital_stock_growth: perpetual-inventory alternative": "no builder and not cited",
}


# ----------------------------------------------------------------------------------------------
# Tree utilities (used by the engine, cell 5b, 6c, 7a and 7b)
def iter_leaves(tree_dict, path=(), eff_weight=1.0):
    """Yield (path, components-or-{'reference':...}, effective_weight) for every leaf."""
    for name, node in tree_dict.items():
        curr_path = path + (name,)
        w = eff_weight * node.get("weight", 1.0)
        if "reference" in node:
            yield "/".join(curr_path), {"reference": node["reference"]}, w
        elif "components" in node:
            yield "/".join(curr_path), node["components"], w
        elif "children" in node:
            yield from iter_leaves(node["children"], curr_path, w)

def iter_leaf_nodes(tree_dict, path=(), eff_weight=1.0):
    """Like iter_leaves but yields the leaf node dict (so callers can read provenance / size_share)."""
    for name, node in tree_dict.items():
        curr_path = path + (name,)
        w = eff_weight * node.get("weight", 1.0)
        if "components" in node:
            yield "/".join(curr_path), node, w
        elif "children" in node:
            yield from iter_leaf_nodes(node["children"], curr_path, w)

def check_tree_weights(tree_dict):
    """Return [(path, sum)] for every sibling group or component list whose weights do not sum to 1."""
    out = []
    def walk(node, p):
        if "children" in node:
            t = sum(ch.get("weight", 0.0) for ch in node["children"].values())
            if abs(t - 1.0) > 1e-3:
                out.append((p, round(t, 4)))
            for k, ch in node["children"].items():
                walk(ch, f"{p}/{k}")
        elif "components" in node:
            t = sum(c.get("weight", 0.0) for c in node["components"])
            if abs(t - 1.0) > 1e-3:
                out.append((p + " (components)", round(t, 4)))
    for k, root in tree_dict.items():
        walk(root, k)
    return out

KINDED_LEAVES = [p for p, n, _ in iter_leaf_nodes(RAY_DALIO_GAUGE_TREES) if "size_share" in n]
print(f"Tree: {len(RAY_DALIO_GAUGE_TREES)} top-level gauges, "
      f"{sum(len(c) for _, c, _ in iter_leaves(RAY_DALIO_GAUGE_TREES) if isinstance(c, list))} components, "
      f"{len(KINDED_LEAVES)} with a size/intensity split.")
print("Weight sums that are not 1.0:", check_tree_weights(RAY_DALIO_GAUGE_TREES) or "none")

# Component kinds for review (flip any `kind=` in the tree above if you disagree)
df_kinds = pd.DataFrame([
    {"gauge_path": p, "component": c["name"], "kind": c["kind"], "weight": round(c["weight"], 4), "primary_code": c["alts"][0].code}
    for p, n, _ in iter_leaf_nodes(RAY_DALIO_GAUGE_TREES) if "size_share" in n for c in n["components"]])
display(df_kinds)

In [ ]:
# 5b. Appendix Sources, Ledger (derived from the tree + data) and Retention
# APPENDIX_SOURCES = the sources/indicators the Appendix names. Ids 1-64 are the A.7 data-source list (pp.183-184);
# string ids are concepts the Appendix names elsewhere (page given). Components cite these via `cites=[...]` in cell 5.
# Last field: reason override for rows no component consumes (default NO_CONSUMER = "feeds no gauge in this tree").
APPENDIX_SOURCES = [
    (1, 'Access to essential health services', 'Social Progress Imperative', 183),
    (2, 'Aerospace and defense sector returns', 'Thomson Reuters', 183),
    (3, 'Agriculture, forestry, fishing value added', 'WDI', 183),
    (4, 'AI: new companies, patents, citations', 'CSET', 183),
    (5, 'Banking claims by currency and source country', 'BIS', 183),
    (6, 'Bloomberg Innovation Index', 'Bloomberg', 183),
    (7, 'Book titles per capita', 'CLIO Infra', 183),
    (8, 'Child mortality rate', 'Social Progress Imperative', 183),
    (9, 'Climate vulnerability and preparedness, subcomponents', 'ND-GAIN', 183),
    (10, 'CO2 emissions', 'CLIO Infra', 183),
    (11, 'Conflict intensity and count per capita', 'Cline Center', 183),
    (12, 'Construction permits score', 'WB Doing Business', 183),
    (13, 'Days lost to labor disputes', 'CLIO Infra', 183),
    (14, 'Discontent, populism, conflict terms in media', 'GDELT', 183),
    (15, 'Education spending', 'WDI', 183),
    (16, 'Educational attainment', 'Barro-Lee', 183),
    (17, 'Electricity production', 'NBER CHAT', 183),
    (18, 'Energy production and consumption', 'EIA', 183),
    (19, 'Enforcing contracts score', 'WB Doing Business', 183),
    (20, 'GINI coefficient', 'WDI', 183),
    (21, 'Google searches for political conflict terms', 'Google Trends', 183),
    (22, 'Google searches indicating discontent', 'Google Trends', 183),
    (23, 'Government disapproval, verbal opposition in news', 'GDELT', 183),
    (24, 'Gross expenditure on R&D', 'WIPO GII', 183),
    (25, 'Historical aviation passengers and cargo', 'NBER CHAT', 183),
    (26, 'Historical technology adoption', 'NBER CHAT', 183),
    (27, 'Homicide rate', 'Social Progress Imperative', 183),
    (28, 'Hours worked per week', 'World Inequality Database', 183),
    (29, 'Human Capital Index', 'World Bank', 183),
    (30, 'Income distributions', 'World Bank', 183),
    (31, 'Internal armed conflict', 'CLIO Infra', 183),
    (32, 'Internet users', 'NBER CHAT', 183),
    (33, 'Male labor force participation', 'World Bank', 183),
    (34, 'Metals and ores production', 'CLIO Infra', 183),
    (35, 'Nuclear weapon stockpile', 'Federation of American Scientists', 183),
    (36, 'Number of new patents', 'WIPO GII', 184),
    (37, 'Numeracy', 'CLIO Infra', 184),
    (38, 'OTC FX turnover', 'BIS', 184),
    (39, 'Particulate matter pollution', 'Social Progress Imperative', 184),
    (40, 'Patent applications', 'WDI', 184),
    (41, 'People displaced by conflict and violence', 'WDI', 184),
    (42, 'People using safely managed drinking water', 'World Bank', 184),
    (43, 'Prevalence of undernourishment', 'World Bank', 184),
    (44, 'Protecting investors value score', 'WB Doing Business', 184),
    (45, 'Protest intensity index', 'GDELT', 184),
    (46, 'Public sector corruption and theft', 'V-Dem', 184),
    (47, 'Rail lines opened and rail cargo', 'NBER CHAT', 184),
    (48, 'Real GDP per worker', 'Penn World Table', 184),
    (49, 'Real growth rate', 'World Bank', 184),
    (50, 'Renewable freshwater resources', 'WDI', 184),
    (51, 'Rigorous and impartial government score', 'V-Dem', 184),
    (52, 'Royalty and license fees', 'WIPO GII', 184),
    (53, 'Share of universities in top 500', 'Shanghai Ranking', 184),
    (54, 'Shipping tonnage by sail, steam, motor and total', 'NBER CHAT', 184),
    (55, 'Starting a business score', 'WB Doing Business', 184),
    (56, 'Suicide mortality rate', 'World Bank', 184),
    (57, 'Tariff rate', 'World Bank', 184),
    (58, 'Top 1% and top 10% income shares', 'World Inequality Database', 184),
    (59, 'Top 1% and top 10% wealth shares', 'World Inequality Database', 184),
    (60, 'Total labor disputes', 'CLIO Infra', 184),
    (61, 'Transparent laws with predictable enforcement', 'V-Dem', 184),
    (62, 'Universities founded', 'CLIO Infra', 184),
    (63, 'Number of new businesses, people creating businesses', 'WIPO GII', 184),
    (64, 'Number of new major websites', 'WIPO GII', 184),
    ("WVS", "World Values Survey items (8)", "World Values Survey", 177),
    ("GEM", "GEM fear of failure & entrepreneurship", "GEM", 179),
    ("GFCI", "Financial Center Index", "Z/Yen", 132),
    ("VACATION", "Vacation days", "ILO / National", 181),
    ("RETIRE", "Effective retirement age", "OECD", 181),
    ("EPL", "Employment protection / hiring-firing rigidity", "OECD", 181),
    ("WAGE", "Wages by education cohort", "ILO / OECD", 162),
    ("RATING", "Sovereign credit rating", "Rating agencies", 133),
    ("DEBT_FOR", "Foreign-held debt share", "IMF / National", 133),
    ("VC", "Venture capital funding", "VentureSource", 132),
    ("INFRA", "Infrastructure quality index", "WEF", 132),
    ("BIS_FX", "FX / debt-securities currency shares", "BIS", 133),
    ("WIPO_GII", "WIPO GII overall score", "WIPO", 132),
    ("OECD_SAV", "Household savings rate", "OECD", 158),
    ("ILO_CB", "Collective bargaining coverage", "ILO", 181),
    ("ILO_MINW", "Minimum wage ratio", "ILO", 181),
    ("COFER", "Currency share of official reserves", "IMF COFER", 133),
    ("WHR_AFF", "Social support & positive affect surveys", "World Happiness Report", 136),
    ("COVID", "COVID outcomes (acts of nature)", "OWID", 133),
]
SOURCE_REASON = {21: "OWNER_SKIP", 22: "OWNER_SKIP"}          # Google Trends skipped per D4; everything else unconsumed = NO_CONSUMER

CARRIED_SERIES = {"DERIVED.WORKING_AGE_GROWTH_FWD10", "DERIVED.WORKING_AGE_GROWTH_YOY"}   # no leaf uses them; kept for the 4(b) check
DERIVED_INPUTS = {                                                  # inputs of build_derived (cell 3)
    "SP.POP.1564.TO", "PROJ.SP.POP.1564.TO", "SP.POP.DPND", "PROJ.SP.POP.DPND", "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG",
    "NE.GDI.FTOT.ZS", "UNHCR.DISPLACED_TOTAL", "SP.POP.TOTL", "PWT.AVH", "LO.PISA.MAT", "LO.PISA.REA", "LO.PISA.SCI",
    "GB.XPD.RSDV.GD.ZS", "NY.GDP.MKTP.CD", "SP.POP.SCIE.RD.P6", "MS.MIL.XPND.CD", "MS.MIL.TOTL.P1", "IP.PAT.RESD",
    "BX.GSR.GNFS.CD", "ARWU.TOP100_COUNT", "TX.VAL.MRCH.CD.WT", "TM.VAL.MRCH.CD.WT", "TX.VAL.FUEL.ZS.UN", "TM.VAL.FUEL.ZS.UN",
    "TX.VAL.AGRI.ZS.UN", "TM.VAL.AGRI.ZS.UN", "TX.VAL.FOOD.ZS.UN", "TM.VAL.FOOD.ZS.UN", "TX.VAL.MMTL.ZS.UN", "TM.VAL.MMTL.ZS.UN",
    "BIS.TC.N.PCT_GDP", "FS.AST.PRVT.GD.ZS", "IMF.GGXWDG_NGDP", "EIA.ENERGY.PRODUCTION_QBTU", "EIA.ENERGY.CONSUMPTION_QBTU",
}
TREE_CODES = {a.code for _, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES) if isinstance(comps, list) for c in comps for a in c["alts"]}
KEEP_CODES = TREE_CODES | DERIVED_INPUTS | CARRIED_SERIES          # cited-only rule: no consumer, no row

# Retention: keep the pre-retention frame for the audit; safe to re-run without re-running cell 4.
if not df_master.attrs.get("retained"):
    df_full = df_master
df_master = df_full[df_full["indicator_code"].isin(KEEP_CODES)].reset_index(drop=True)
df_master.attrs["retained"] = True
_trimmed = (df_full[~df_full["indicator_code"].isin(KEEP_CODES)].groupby("indicator_code")["country_code"].nunique().sort_values(ascending=False))
print(f"Retention: kept {df_master['indicator_code'].nunique()} series ({len(df_master):,} rows); trimmed {len(_trimmed)} with no consumer.")
_expected_derived = {c for c in TREE_CODES if c.startswith("DERIVED.") or c == "EIA.ENERGY.SELF_SUFF"} | CARRIED_SERIES
_missing = sorted(_expected_derived - set(df_master["indicator_code"]))
print("Derived series the tree or checks expect but that were not built:", _missing or "none")

# Ledger: one row per APPENDIX_SOURCES entry. Status comes from the data and the tree, never from a hand-written field.
_TIER_STATUS = {"exact": "live", "substitute_source": "substitute", "proxy_concept": "proxy"}
_TIER_ORDER = {"exact": 0, "substitute_source": 1, "proxy_concept": 2}
_cov = df_master.groupby("indicator_code").agg(n=("country_code", "nunique"), last=("year", "max"))
_consumers = defaultdict(list)
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            for _i in _c["cites"]:
                _consumers[_i].append((_lp, _c))

def build_ledger():
    rows = []
    for sid, indicator, source, page in APPENDIX_SOURCES:
        cons = _consumers.get(sid, [])
        alts = [a for _, c in cons for a in c["alts"]]
        live = [a for a in alts if a.code in _cov.index]
        if not cons:
            status, reason = "skipped", SOURCE_REASON.get(sid, "NO_CONSUMER")
        elif live:
            best = min(live, key=lambda a: _TIER_ORDER[a.tier])
            status, reason = _TIER_STATUS[best.tier], ""
        else:
            gaps = [a for a in alts if a.status == "GAP"]
            status, reason = ("gap", gaps[0].reason) if gaps else ("dead", "NO_ROWS")
        rows.append({"id": sid, "indicator": indicator, "source": source, "page": page,
                     "components": len(cons), "codes_with_data": ", ".join(sorted({a.code for a in live})),
                     "n_countries": max((int(_cov.loc[a.code, "n"]) for a in live), default=0),
                     "latest_year": max((int(_cov.loc[a.code, "last"]) for a in live), default=np.nan),
                     "status": status, "reason": reason})
    return pd.DataFrame(rows)

df_ledger = build_ledger()
print(df_ledger["status"].value_counts().to_string())
print("reasons:", df_ledger.loc[df_ledger["reason"] != "", "reason"].value_counts().to_dict())
display(df_ledger)

# 6. Minimal Scoring Engine (§6)
Computes annual gauge scores from `YEAR_FLOOR = 1950` to the last actual year.
Steps:
1. Wide country × year matrices.
2. Standardisation: percentile rank within (code, year) across countries; ties averaged, scaled 0–100.
   Valid only if cross-section count ≥ `MIN_CROSS_SECTION = 30`.
3. Coalescing (D16): first available alternative supplies value. Proxy fallbacks excluded unless `ALLOW_PROXY_FALLBACK=True`.
4. Bottom-up renormalisation (D20): missing components/children are renormalised over present children.
5. Reference resolution: `@dotted.path` references evaluated before consuming nodes.
6. Exports parquet files to `data/dalitest/out/`.


In [ ]:
# 6. Minimal Scoring Engine (§6)
def _stack_all(m):
    try:
        return m.stack(future_stack=True)
    except TypeError:                                    # older pandas
        return m.stack(dropna=False)


def run_gauge_engine(df, trees, year_floor=1950, allow_proxy_fallback=False,
                     min_cross_section=30, universe=None, write_outputs=True):
    countries = list(UNIVERSE if universe is None else universe)
    if not df.empty and "is_projection" in df.columns:
        non_proj = df[~df["is_projection"].astype(bool)]
    else:
        non_proj = df
    if not non_proj.empty:
        present_years = set(non_proj["year"].unique())
        last_actual_year = int(non_proj["year"].max())
        years = sorted(y for y in range(year_floor, last_actual_year + 1) if y in present_years)
    else:
        years = []
    if not years:
        years = [2024]

    # 1. Wide matrices (indicator -> DataFrame[index=years, columns=countries])
    wide = {}
    if not non_proj.empty and "indicator_code" in non_proj.columns:
        for code, g in non_proj.groupby("indicator_code"):
            w = g.pivot_table(index="year", columns="country_code", values="value", aggfunc="first")
            wide[code] = w.reindex(index=years, columns=countries)
    empty_mat = pd.DataFrame(np.nan, index=years, columns=countries)

    # 2. Ranks per (code, direction), only for codes the tree uses
    code_dirs = defaultdict(set)
    for _, comps, _ in iter_leaves(trees):
        if isinstance(comps, list):
            for c in comps:
                for a in c.get("alts", []):
                    code_dirs[a.code].add(a.direction)

    ranked_mats = {}
    rank_support_rows = []
    for code, w in wide.items():
        counts = w.count(axis=1)
        for y in years:
            rank_support_rows.append({"indicator_code": code, "year": y, "n_ranked": int(counts.get(y, 0))})
        for d in code_dirs.get(code, ()):
            directed = w * d
            r_df = empty_mat.copy()
            for y in years:
                n = int(counts.get(y, 0))
                if n >= min_cross_section:
                    row = directed.loc[y].dropna()
                    rk = row.rank(method="average", ascending=True)
                    r_df.loc[y, rk.index] = (rk - 1.0) / (n - 1.0) * 100.0
            ranked_mats[(code, d)] = r_df
    df_rank_support = pd.DataFrame(rank_support_rows, columns=["indicator_code", "year", "n_ranked"])

    # 3. Component coalescing and leaf scores
    comp_score_records, leaf_scores, leaf_w_pres = [], {}, {}
    for leaf_path, comps, _ in iter_leaves(trees):
        if not isinstance(comps, list):
            continue
        scores, weights = [], []
        for comp in comps:
            alts = comp.get("alts", [])
            c_score = empty_mat.copy()
            c_alt_used = pd.DataFrame(-1, index=years, columns=countries)
            primary_seen = False
            for a_idx, a in enumerate(alts):
                if a.code not in wide:                  # no data anywhere: skipped when choosing the primary
                    continue
                is_primary = not primary_seen
                primary_seen = True
                eligible = is_primary or a.tier in ("exact", "substitute_source") or allow_proxy_fallback
                if not eligible:
                    continue
                r_mat = ranked_mats.get((a.code, a.direction))
                if r_mat is None:
                    continue
                fill = c_score.isna() & r_mat.notna()
                c_score = c_score.mask(fill, r_mat)
                c_alt_used = c_alt_used.mask(fill, a_idx)
            scores.append(c_score)
            weights.append(comp.get("weight", 1.0))

            st = _stack_all(c_score).reset_index()
            st.columns = ["year", "country_code", "score"]
            st["alt_used"] = _stack_all(c_alt_used).values
            st["observed"] = st["score"].notna()
            st["leaf_path"] = leaf_path
            st["component_name"] = comp["name"]
            comp_score_records.append(st)

        num = pd.DataFrame(0.0, index=years, columns=countries)
        den = num.copy()
        for wt, s in zip(weights, scores):
            num += wt * s.fillna(0.0)
            den += wt * s.notna().astype(float)
        leaf_scores[leaf_path] = num / den.replace(0.0, np.nan)
        leaf_w_pres[leaf_path] = den / sum(weights)

    comp_cols = ["country_code", "year", "leaf_path", "component_name", "score", "alt_used", "observed"]
    df_comp_scores = (pd.concat(comp_score_records, ignore_index=True)[comp_cols]
                      if comp_score_records else pd.DataFrame(columns=comp_cols))

    # 4. Bottom-up node scores; references are evaluated on demand
    node_scores, node_w_pres, node_records = {}, {}, []
    done_trees, in_progress = set(), set()

    def eval_tree(name):
        if name in done_trees:
            return
        if name in in_progress:
            raise ValueError(f"Circular @reference through tree '{name}'")
        in_progress.add(name)
        walk_eval(trees[name], name)
        in_progress.discard(name)
        done_trees.add(name)

    def walk_eval(node, path_str):
        if "reference" in node:
            ref_path = "/".join(node["reference"].lstrip("@").split("."))
            top = ref_path.split("/")[0]
            if top not in done_trees:
                eval_tree(top)
            s, wp = node_scores[ref_path], node_w_pres[ref_path]
        elif "components" in node:
            s, wp = leaf_scores[path_str], leaf_w_pres[path_str]
        else:
            ch_s, ch_w, ch_wp = [], [], []
            for ch_name, ch_node in node["children"].items():
                cs, cwp = walk_eval(ch_node, f"{path_str}/{ch_name}")
                ch_s.append(cs); ch_w.append(ch_node.get("weight", 1.0)); ch_wp.append(cwp)
            num = pd.DataFrame(0.0, index=years, columns=countries)
            den, wpn = num.copy(), num.copy()
            for wt, cs, cwp in zip(ch_w, ch_s, ch_wp):
                num += wt * cs.fillna(0.0)
                den += wt * cs.notna().astype(float)
                wpn += wt * cwp.fillna(0.0)
            s, wp = num / den.replace(0.0, np.nan), wpn / sum(ch_w)
        node_scores[path_str], node_w_pres[path_str] = s, wp
        rec = _stack_all(s).reset_index()
        rec.columns = ["year", "country_code", "score"]
        rec["weight_present"] = _stack_all(wp).values
        rec["node_path"] = path_str
        node_records.append(rec)
        return s, wp

    for tree_name in trees:
        eval_tree(tree_name)

    node_cols = ["country_code", "year", "node_path", "score", "weight_present"]
    df_node_scores = (pd.concat(node_records, ignore_index=True)[node_cols]
                      if node_records else pd.DataFrame(columns=node_cols))

    if write_outputs:                                    # FRD §6.2
        df_comp_scores.to_parquet(OUT_DIR / "component_scores.parquet", index=False)
        df_node_scores.to_parquet(OUT_DIR / "node_scores.parquet", index=False)
        df_rank_support.to_parquet(OUT_DIR / "rank_support.parquet", index=False)
        print("Engine complete! Outputs written to data/dalitest/out/.")
    return df_comp_scores, df_node_scores, df_rank_support


df_comp_scores, df_node_scores, df_rank_support = run_gauge_engine(
    df_master, RAY_DALIO_GAUGE_TREES, year_floor=YEAR_FLOOR,
    allow_proxy_fallback=ALLOW_PROXY_FALLBACK, min_cross_section=MIN_CROSS_SECTION)

In [ ]:
# 6b. Engine self-test (runs the REAL engine)
def _engine_selftest():
    cs = [f"C{i:02d}" for i in range(60)]
    yrs = [2018, 2019, 2020]
    rows = []

    def add(code, fn, years=yrs, only=None):
        for y in years:
            for i, c in enumerate(cs):
                if only is None or i in only:
                    rows.append({"country_code": c, "indicator_code": code, "year": y,
                                 "value": fn(i), "source": "T", "is_projection": False})

    add("T.GDPPC", lambda i: 1000.0 + 100 * i)                         # richer = higher index
    add("T.PRIMARY", lambda i: float(i), only=set(range(0, 40)))       # 40 countries
    add("T.SUBST", lambda i: 100.0 - i, only=set(range(0, 50)))        # 50 countries
    add("T.PROXY", lambda i: float(i))                                 # all 60
    add("T.THIN", lambda i: float(i), years=[2020], only=set(range(0, 20)))   # < min cross-section
    df = pd.DataFrame(rows)
    A = alt
    trees = {
        "prod": {"weight": 1.0, "children": {"cost": {"weight": 1.0, "components": [
            {"name": "c", "weight": 1.0, "alts": [A("T.GDPPC", "exact", -1)]}]}}},
        "strength": {"weight": 1.0, "components": [
            {"name": "pc", "weight": 1.0, "alts": [A("T.GDPPC", "exact", +1)]}]},
        "coal": {"weight": 1.0, "components": [
            {"name": "x", "weight": 0.75, "alts": [A("T.PRIMARY", "exact", +1),
                                                   A("T.SUBST", "substitute_source", +1),
                                                   A("T.PROXY", "proxy_concept", +1)]},
            {"name": "y", "weight": 0.25, "alts": [A("T.THIN", "exact", +1)]}]},
        "skip1": {"weight": 1.0, "components": [
            {"name": "z", "weight": 1.0, "alts": [A("T.NOPE", "exact", +1),
                                                   A("T.PROXY", "proxy_concept", +1)]}]},
        "viaref": {"weight": 1.0, "children": {"a": {"weight": 1.0, "reference": "@prod.cost"}}},
    }
    kw = dict(year_floor=2018, min_cross_section=30, universe=cs, write_outputs=False)
    comp, node, supp = run_gauge_engine(df, trees, **kw)

    def ns(path, y):   return node[(node.node_path == path) & (node.year == y)].set_index("country_code")
    def cmp_(leaf, name, y): return comp[(comp.leaf_path == leaf) & (comp.component_name == name)
                                        & (comp.year == y)].set_index("country_code")

    # (1) direction is per leaf: the same code ranks in opposite directions in two leaves
    cost, strength = ns("prod/cost", 2019)["score"], ns("strength", 2019)["score"]
    assert cost.idxmin() == "C59" and cost.idxmax() == "C00" and cost.min() == 0 and cost.max() == 100
    assert strength.idxmax() == "C59" and strength.idxmin() == "C00"
    assert np.allclose(cost + strength, 100.0)

    # (2) references resolve and equal their target
    assert np.allclose(ns("viaref/a", 2019)["score"], cost, equal_nan=True)

    # (3) coalescing: exact primary, then substitute_source; proxy excluded by default
    x = cmp_("coal", "x", 2020)
    assert (x.loc[[f"C{i:02d}" for i in range(40)], "alt_used"] == 0).all()
    assert (x.loc[[f"C{i:02d}" for i in range(40, 50)], "alt_used"] == 1).all()
    assert x.loc[[f"C{i:02d}" for i in range(50, 60)], "score"].isna().all()
    assert np.isclose(x.loc["C05", "score"], 5 / 39 * 100)               # rank among the 40 reporters
    assert np.isclose(x.loc["C45", "score"], (49 - 45) / 49 * 100)       # rank among the 50 (reversed scale)

    # (4) proxy fallback only with the switch
    comp2, _, _ = run_gauge_engine(df, trees, allow_proxy_fallback=True, **kw)
    x2 = comp2[(comp2.leaf_path == "coal") & (comp2.component_name == "x") & (comp2.year == 2020)].set_index("country_code")
    assert (x2.loc[[f"C{i:02d}" for i in range(50, 60)], "alt_used"] == 2).all()

    # (5) renormalisation and weight_present: y has no valid ranks (20 < 30), so x carries the leaf
    n2020 = ns("coal", 2020)
    assert np.isclose(n2020.loc["C05", "weight_present"], 0.75)
    assert np.isclose(n2020.loc["C05", "score"], x.loc["C05", "score"])
    assert n2020.loc["C55", "weight_present"] == 0 and np.isnan(n2020.loc["C55", "score"])
    assert supp[(supp.indicator_code == "T.THIN") & (supp.year == 2020)]["n_ranked"].iloc[0] == 20

    # (6) a dead first alternative is skipped; the next one becomes primary (even a proxy) and keeps its list index
    z = cmp_("skip1", "z", 2019)
    assert z["score"].notna().all() and (z["alt_used"] == 1).all()
    print("Engine self-test passed (direction per leaf, refs, coalescing, proxy switch, renormalisation, dead primary).")


_engine_selftest()

In [ ]:
# 6c. Quality-only views (post-hoc, from df_comp_scores; the engine is unchanged).
# Total view = the node's own score in df_node_scores. Quality-only = the `intensity` components only, weights renormalised
# over those present. Cost Competitiveness and Reserve Status have no Quality-only view (no kind tags).
def quality_only_scores(df_comp, trees):
    out = []
    for path, node, _ in iter_leaf_nodes(trees):
        if "size_share" not in node:
            continue
        w = {c["name"]: c["weight"] for c in node["components"] if c["kind"] == "intensity"}
        sub = df_comp[(df_comp["leaf_path"] == path) & df_comp["component_name"].isin(w) & df_comp["score"].notna()].copy()
        sub["w"] = sub["component_name"].map(w)
        sub["ws"] = sub["score"] * sub["w"]
        g = sub.groupby(["country_code", "year"], as_index=False).agg(ws=("ws", "sum"), w=("w", "sum"))
        g["score"] = g["ws"] / g["w"]
        g["weight_present"] = g["w"] / sum(w.values())
        g["gauge_path"] = path
        out.append(g[["country_code", "year", "gauge_path", "score", "weight_present"]])
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["country_code", "year", "gauge_path", "score", "weight_present"])

df_quality_scores = quality_only_scores(df_comp_scores, RAY_DALIO_GAUGE_TREES)

_tot = df_node_scores[df_node_scores["node_path"].isin(KINDED_LEAVES)].rename(columns={"node_path": "gauge_path", "score": "total"})
df_views = _tot.merge(df_quality_scores.rename(columns={"score": "quality_only", "weight_present": "wp_quality"}),
                      on=["country_code", "year", "gauge_path"], how="left")
_v = df_views[(df_views["year"] == ASOF_YEAR) & df_views["country_code"].isin(DALIO_24)]
print(f"Total vs Quality-only at {ASOF_YEAR} (Dalio-24), 0-100 percent ranks:")
_v = _v.assign(gauge=_v["gauge_path"].str.split("/").str[-1])
display(_v.pivot_table(index="country_code", columns="gauge", values=["total", "quality_only"]).round(0))

# 7a. QA Reports & PDF Ledger (§7.1)
Generates all diagnostics specified in FRD §7.1 and exports to `data/dalitest/out/`:
- `df_health`: indicator-level health with PPP GDP weights, tier, stale flag, first valid year.
- `pdf_ledger.csv`: complete ledger of 64 Appendix items + §11 non-list inputs.
- `df_registry.csv`: series registry with consumer status.
- `alt_coverage.csv`: alternative coverage by leaf and component.
- `effective_weights.csv`: effective weights and tier/provenance distribution.
- `switch_report.csv`: alternative switch counts across countries.
- `agreement_report.csv`: empirical Spearman correlation between alternative pairs.
- `proxy_candidates.csv`: coverage impact of excluded proxies.
- `direction_map.csv`: complete direction map with ambiguous flags.


In [ ]:
# 7a. QA Reports (displayed, not saved). Re-runnable after cells 5b and 6.
from IPython.display import display

def show(title, df, n=60):
    print(f"\n=== {title} ({len(df)} rows) ===")
    display(df.head(n) if n else df)

wide_map = {code: g.set_index(["year", "country_code"])["value"] for code, g in df_master.groupby("indicator_code")}
code_leaf_map, _tier_sets = defaultdict(list), defaultdict(set)
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            for _a in _c["alts"]:
                code_leaf_map[_a.code].append(_lp)
                _tier_sets[_a.code].add(_a.tier)

def _gauge(leaf_path):                              # strength determinants are reported individually
    p = leaf_path.split("/")
    return p[1] if p[0] == "strength_determinants" else p[0]

# 1. Series health: coverage per indicator (universe-wide, target-set, observations-weighted)
health_rows = []
for code, g in df_master.groupby("indicator_code"):
    actuals = g[~g["is_projection"]]
    ev = actuals if not actuals.empty else g
    all_rep = set(ev["country_code"].unique())
    recent = set(ev[ev["year"] >= STALE_BEFORE]["country_code"].unique())
    cs = ev.groupby("year")["country_code"].nunique()
    valid = cs[cs >= MIN_CROSS_SECTION].index
    leaves = code_leaf_map.get(code, [])
    health_rows.append({
        "gauge": _gauge(leaves[0]) if leaves else "unassigned", "indicator": code, "source": g["source"].iloc[0],
        "countries": len(all_rep), "global_cov_%": round(len(all_rep) / len(UNIVERSE) * 100, 1),
        "weighted_cov_%": round(OBS_WEIGHTS[list(all_rep)].sum() * 100, 1),
        f"weighted_recent_% (>={STALE_BEFORE})": round(OBS_WEIGHTS[list(recent)].sum() * 100, 1),
        "latest_year": int(ev["year"].max()), "first_valid_year": int(valid.min()) if len(valid) else np.nan,
        "degenerate": bool(ev["value"].nunique() < 5), "has_projection": bool(g["is_projection"].any()),
    })
df_health = pd.DataFrame(health_rows).sort_values(["gauge", "weighted_cov_%"], ascending=[True, False])
show("Series health", df_health, None)

# 2. Registry: why every retained series is in the master table
def classify_series(code):
    if code in code_leaf_map:
        return "stand_in" if _tier_sets[code] == {"proxy_concept"} else "wired"
    if code in CARRIED_SERIES:
        return "carried"
    return "derived_input" if code in DERIVED_INPUTS else "no_consumer"
df_registry = (df_master.groupby("indicator_code").agg(source=("source", "first"), n_countries=("country_code", "nunique"),
               first_year=("year", "min"), last_year=("year", "max")).reset_index().rename(columns={"indicator_code": "code"}))
df_registry["status"] = df_registry["code"].map(classify_series)
df_registry["leaves"] = df_registry["code"].map(lambda c: "; ".join(sorted(set(code_leaf_map.get(c, [])))))
print("\nRegistry status counts:", df_registry["status"].value_counts().to_dict())

# 3. Alternative coverage (dead = in the tree, not a declared GAP, no data)
cov_rows = []
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            for i, a in enumerate(_c["alts"]):
                g = wide_map.get(a.code)
                rep = set(g.index.get_level_values("country_code")) if g is not None else set()
                cov_rows.append({"leaf": _lp, "component": _c["name"], "alt_index": i, "code": a.code, "tier": a.tier,
                                 "countries": len(rep), "weighted_cov_%": round(OBS_WEIGHTS[list(rep)].sum() * 100, 1) if rep else 0.0,
                                 "gap": a.status == "GAP", "reason": a.reason or "", "dead": (not rep) and a.status != "GAP"})
df_alt_cov = pd.DataFrame(cov_rows)
show("Dead alternatives (in the tree, not a declared GAP, no data)", df_alt_cov[df_alt_cov["dead"]], None)

# 4. Effective weights: static share of each gauge's nominal weight by state, tier, basis, kind
eff_rows = []
for _lp, _node, _lw in iter_leaf_nodes(RAY_DALIO_GAUGE_TREES):
    for _c in _node["components"]:
        alts = _c["alts"]
        primary = next((a for a in alts if a.code in wide_map), None)
        if primary is not None:
            state, tier, code = "live", primary.tier, primary.code
        elif alts[0].status == "GAP":
            state, tier, code = "gap", "gap", alts[0].code
        else:
            state, tier, code = "dead", "gap", alts[0].code
        eff_rows.append({"gauge": _gauge(_lp), "leaf_path": _lp, "component": _c["name"], "kind": _c.get("kind", ""),
                         "basis": _c["basis"], "page": _c["page"], "cites": _c["cites"],
                         "weight": _c["weight"], "effective_weight": _lw * _c["weight"], "weights": _node.get("provenance", ""),
                         "state": state, "tier": tier, "primary_code": code,
                         "alts": " > ".join(f"{a.code}[{a.tier[:3]},{a.direction:+d}" + (f",{a.reason}" if a.status == "GAP" else "") + "]" for a in alts)})
df_eff = pd.DataFrame(eff_rows)          # also the portable tree export: one row per component with weight, kind, basis, page, cites, alternatives
show("Tree export / effective weights", df_eff, 25)
def _share(col):
    t = df_eff.pivot_table(index="gauge", columns=col, values="effective_weight", aggfunc="sum", fill_value=0.0)
    return t.div(t.sum(axis=1), axis=0).round(3)
eff_summary = pd.concat({"by_state": _share("state"), "by_tier": _share("tier"), "by_basis": _share("basis")}, axis=1)
print("\nShare of each gauge's nominal weight (competitiveness is a reference and is not expanded here):")
display(eff_summary)

# 5. Direction map: components whose sign deserves a second look
dir_rows = []
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            for a in _c["alts"]:
                amb = any(k in a.code for k in ["CPI", "INFLATION", "CBPOL", "TRFT", "TOTL.GD.ZS", "EPL", "COLLECTIVE"]) or ("cost" in _c["name"] and "GDP" in a.code)
                dir_rows.append({"code": a.code, "leaf_path": _lp, "direction": a.direction, "ambiguous": amb, "tier": a.tier})
df_dir = pd.DataFrame(dir_rows).drop_duplicates(["code", "leaf_path"])
show("Direction map: ambiguous signs", df_dir[df_dir["ambiguous"]], None)

# 6. Alternatives switched mid-series (needs df_comp_scores from cell 6)
obs = df_comp_scores[df_comp_scores["observed"]]
df_switch = obs.groupby(["leaf_path", "component_name", "alt_used"]).size().rename("country_years").reset_index()
_n_alt = obs.groupby(["leaf_path", "component_name", "country_code"])["alt_used"].nunique()
_sw = (_n_alt[_n_alt > 1].reset_index().groupby(["leaf_path", "component_name"]).agg(total_switched=("country_code", "count"),
       target_switched=("country_code", lambda s: int(s.isin(TARGET_SET).sum()))).reset_index())
df_switch = df_switch.merge(_sw, on=["leaf_path", "component_name"], how="left").fillna({"total_switched": 0, "target_switched": 0})
show("Alternative switches", df_switch[df_switch["total_switched"] > 0], None)

# 7. Agreement between alternatives of the same component (Spearman on overlapping country-years)
agree = []
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            alts = _c["alts"]
            for i in range(len(alts)):
                for j in range(i + 1, len(alts)):
                    c1, c2 = alts[i].code, alts[j].code
                    if c1 in wide_map and c2 in wide_map:
                        al = pd.concat([wide_map[c1], wide_map[c2]], axis=1, keys=["v1", "v2"]).dropna()
                        tg = al[al.index.get_level_values("country_code").isin(TARGET_SET)]
                        agree.append({"leaf_path": _lp, "component": _c["name"], "alt_1": c1, "alt_2": c2, "overlap": len(al), "target_overlap": len(tg),
                                      "spearman_global": round(spearmanr(al["v1"], al["v2"])[0], 3) if len(al) >= 5 else np.nan,
                                      "spearman_target": round(spearmanr(tg["v1"], tg["v2"])[0], 3) if len(tg) >= 5 else np.nan})
df_agreement = pd.DataFrame(agree)
show("Agreement between alternatives", df_agreement, None)

# 8. Proxy candidates: country-years a proxy alternative would add if proxy fallback were allowed
prox = []
for _lp, _comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES):
    if isinstance(_comps, list):
        for _c in _comps:
            for a in _c["alts"][1:]:
                if a.tier == "proxy_concept" and a.code in wide_map:
                    sc = df_comp_scores[(df_comp_scores["leaf_path"] == _lp) & (df_comp_scores["component_name"] == _c["name"])]
                    miss = sc[~sc["observed"]][["year", "country_code"]]
                    idx = wide_map[a.code].index
                    added = [(y, c) for y, c in zip(miss["year"], miss["country_code"]) if (y, c) in idx]
                    prox.append({"leaf_path": _lp, "component": _c["name"], "proxy_code": a.code, "country_years_added": len(added),
                                 "target_years_added": sum(c in TARGET_SET for _, c in added)})
df_proxy = pd.DataFrame(prox)
show("Proxy candidates", df_proxy, None)

# 7b. Inline Asserts & Hard Gates (§7.2, §7.3, §14)
Enforces all Hard Gates (Definition of Done) and structure checks:
- **Gate 1 (Ledger Gate)**: Every item has status; every gap/skipped row has a reason.
- **Gate 2 (Registry Gate)**: `no_consumer` is empty; no dead alternative outside GAP.
- **Gate 3 (Weight Gate)**: Hours-worked effective weight is 15.45% ± 0.1%; health weights sum to 1.0; happiness weights sum to 1.0; growth split 0.65/0.35; productivity split 0.70/0.30; no code in two leaves of one gauge.
- **Gate 4 (Data Sanity Gates)**:
  - 4(a) `IMF.NGDPDPC` for DALIO_35 vs §14.3 (median rel dev ≤ 10%, max ≤ 30%).
  - 4(b) `DERIVED.WORKING_AGE_GROWTH_FWD10` at 2024 for DALIO_35 vs §14.2 (≥ 90% within 0.3 pp).
- **Printed Checks (§14.1, §14.4)**: Headline Empire Scores comparison & qualitative checks.


In [ ]:
# 7b. Checks (print-only: nothing raises, nothing is pass/fail-gated)
def resolve_ref(ref, trees):
    node, parts = None, ref.lstrip("@").split(".")
    node = trees.get(parts[0])
    for p in parts[1:]:
        node = (node or {}).get("children", {}).get(p)
    return node

# --- Structure ---
for g in ["health_index", "happiness_index"]:
    codes = [a.code for c in RAY_DALIO_GAUGE_TREES[g]["components"] for a in c["alts"]]
    if len(codes) != len(set(codes)):
        print(f"NOTE: duplicate codes in {g}: {sorted({c for c in codes if codes.count(c) > 1})}")
unresolved = [(lp, comps["reference"]) for lp, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES)
              if isinstance(comps, dict) and resolve_ref(comps["reference"], RAY_DALIO_GAUGE_TREES) is None]
print("Unresolved @references:", unresolved or "none")
bad_reason = [(c["name"], a.reason) for _, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES) if isinstance(comps, list)
              for c in comps for a in c["alts"] if a.status == "GAP" and a.reason not in GAP_REASONS]
print("GAP alternatives with an unknown reason code:", bad_reason or "none")
_cite_ids = {s[0] for s in APPENDIX_SOURCES}
bad_cites = [(c["name"], i) for _, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES) if isinstance(comps, list)
             for c in comps for i in c["cites"] if i not in _cite_ids]
print("Cites that do not resolve in APPENDIX_SOURCES:", bad_cites or "none")

# --- Weights vs the Appendix tables ---
APPENDIX_WEIGHTS = {                                   # Appendix p.136 (health, happiness) and p.158 (investing), in %
    "health_index": {"life_expectancy": 30.0, "child_mortality": 10.0, "air_pollution": 5.0, "safely_managed_drinking_water": 5.0,
                     "basic_sanitation": 5.0, "food_insecurity": 15.0, "homicide": 7.5, "displaced_population": 7.5, "essential_health_services": 15.0},
    "happiness_index": {"share_reported_happy": 22.5, "share_reported_daily_enjoyment": 22.5, "life_satisfaction": 22.5,
                        "good_social_support_network": 22.5, "suicide_rate": 10.0},
    "productivity_gauge/what_you_pay_vs_what_you_get/investing": {"investment_pct_gdp": 3.1, "household_savings_rate": 3.1, "capital_productivity": 3.1,
                        "capital_stock_growth": 3.1, "income_volatility": 2.6, "inflation_volatility": 2.6},
}
leaf_by_path = {lp: comps for lp, comps, _ in iter_leaves(RAY_DALIO_GAUGE_TREES) if isinstance(comps, list)}
for lp, tbl in APPENDIX_WEIGHTS.items():
    tot = sum(tbl.values())
    diffs = {c["name"]: abs(c["weight"] - tbl[c["name"]] / tot) for c in leaf_by_path[lp] if c["name"] in tbl}
    print(f"Weights vs Appendix, {lp}: {len(diffs)}/{len(tbl)} components matched, max abs diff {max(diffs.values()):.4f}")
fg = RAY_DALIO_GAUGE_TREES["future_growth"]["children"]
pg = RAY_DALIO_GAUGE_TREES["productivity_gauge"]["children"]
print(f"future_growth split {fg['productivity_impact']['weight']}/{fg['indebtedness_impact']['weight']} (Appendix 65/35); "
      f"productivity split {pg['what_you_pay_vs_what_you_get']['weight']}/{pg['culture']['weight']} (Appendix 70/30)")
hours_w = sum(lw * c["weight"] for lp, node, lw in iter_leaf_nodes({"productivity_gauge": RAY_DALIO_GAUGE_TREES["productivity_gauge"]})
              for c in node["components"] if "ILO.HOURS_WEEKLY_MODELLED_MALE" in [a.code for a in c["alts"]])
print(f"Hours-worked effective weight in the productivity gauge: {hours_w*100:.2f}% (Appendix C: 15.45%)")
print("Empire Score determinant weights (normalised):", {k: round(v["weight"], 3) for k, v in RAY_DALIO_GAUGE_TREES["strength_determinants"]["children"].items()})

# --- Ledger and registry ---
print("\nLedger status:", df_ledger["status"].value_counts().to_dict())
print("Ledger rows still on the interim PROBE_NEXT reason (Stage 2 will settle them):", int((df_ledger["reason"] == "PROBE_NEXT").sum()))
print("Ledger rows promised data but none arrived (dead):", df_ledger.loc[df_ledger["status"] == "dead", "id"].tolist() or "none")
print("Dead alternatives:", int(df_alt_cov["dead"].sum()), "| series with no consumer:", int((df_registry["status"] == "no_consumer").sum()))

# --- Data sanity against the Appendix (printed only) ---
PDF_NGDPDPC_35 = {"IND": 2.5, "IDN": 4.8, "ZAF": 6.2, "THA": 7.0, "COL": 7.9, "BRA": 10.0, "ARG": 10.4, "CHN": 12.7, "MEX": 14.8, "TUR": 14.8,
    "RUS": 14.9, "CHL": 16.3, "POL": 22.7, "HUN": 23.0, "GRC": 24.3, "PRT": 29.1, "JPN": 30.8, "CZE": 32.5, "SAU": 33.1, "ESP": 34.5,
    "KOR": 35.6, "ITA": 38.9, "FRA": 47.0, "GBR": 51.2, "ARE": 54.0, "DEU": 55.1, "CAN": 55.3, "BEL": 55.5, "SWE": 57.1, "NLD": 66.6,
    "AUS": 67.6, "USA": 83.1, "NOR": 87.4, "SGP": 91.6, "CHE": 101.1}
PDF_WA_GROWTH_FWD10 = {"IND": 0.9, "ARE": 0.8, "IDN": 0.7, "SAU": 1.6, "TUR": 0.4, "CHN": -0.4, "POL": -0.6, "ZAF": 1.0, "RUS": -0.3, "COL": 0.5,
    "SGP": -0.2, "MEX": 0.8, "CHL": 0.2, "SWE": 0.4, "AUS": 0.6, "ARG": 0.8, "THA": -0.7, "HUN": -0.5, "CZE": 0.0, "KOR": -1.2,
    "BRA": 0.1, "USA": 0.2, "GBR": 0.4, "NLD": -0.2, "JPN": -0.8, "CAN": 0.3, "PRT": -0.8, "BEL": -0.2, "FRA": -0.1, "NOR": 0.1,
    "ESP": -0.7, "CHE": -0.3, "GRC": -0.9, "DEU": -1.1, "ITA": -1.1}
PDF_EMPIRE_SCORES = {"USA": 0.89, "CHN": 0.80, "DEU": 0.38, "JPN": 0.33, "KOR": 0.32, "IND": 0.30, "GBR": 0.29, "FRA": 0.27, "RUS": 0.26,
    "SGP": 0.24, "AUS": 0.23, "TUR": 0.21, "CAN": 0.21, "CHE": 0.19, "BRA": 0.18, "NLD": 0.17, "IDN": 0.17, "ITA": 0.17, "ESP": 0.17,
    "SAU": 0.15, "MEX": 0.14, "ZAF": 0.10, "ARG": 0.07}

def _at(code, year):
    s = df_master[(df_master["indicator_code"] == code) & (df_master["year"] == year)].drop_duplicates("country_code")
    return s.set_index("country_code")["value"]

imf = _at("IMF.NGDPDPC", 2024)
if imf.empty:
    print("4(a) IMF.NGDPDPC 2024: no data")
else:
    dev = {c: abs(float(imf[c]) / 1000 - t) / t for c, t in PDF_NGDPDPC_35.items() if c in imf.index}
    print(f"4(a) NGDP per capita vs Appendix p.161: {len(dev)} countries, median rel dev {np.median(list(dev.values()))*100:.1f}%, "
          f"max {max(dev.values())*100:.1f}% ({max(dev, key=dev.get)})")
wa = _at("DERIVED.WORKING_AGE_GROWTH_FWD10", 2024)
if wa.empty:
    print("4(b) DERIVED.WORKING_AGE_GROWTH_FWD10 2024: no data")
else:
    d = np.array([abs(float(wa[c]) - t) for c, t in PDF_WA_GROWTH_FWD10.items() if c in wa.index])
    print(f"4(b) working-age growth vs Appendix p.148: {len(d)} countries, {(d <= 0.3).mean()*100:.0f}% within 0.3 pp")

emp = df_node_scores[(df_node_scores["node_path"] == "strength_determinants") & (df_node_scores["year"] == ASOF_YEAR)].set_index("country_code")["score"]
common = [c for c in DALIO_24 if c in emp.index and c in PDF_EMPIRE_SCORES and pd.notna(emp[c])]
if len(common) >= 10:
    rho = spearmanr([emp[c] for c in common], [PDF_EMPIRE_SCORES[c] for c in common])[0]
    print(f"Empire Score (Total) at {ASOF_YEAR} vs Appendix p.135: Spearman {rho:.3f} across {len(common)} countries")
else:
    print(f"Empire Score vs Appendix: only {len(common)} comparable countries at {ASOF_YEAR}")